# Module 3: Periodic Action Module (UTH-Based Adjustments)

## Purpose
This module runs at 12 PM, 3 PM, 6 PM, 9 PM, and 12 AM Cairo time to:
1. Adjust prices based on Up-Till-Hour (UTH) performance vs benchmarks
2. Manage SKU discounts and Quantity Discounts based on performance
3. Adjust cart rules dynamically

## UTH Benchmarks
- Calculate historical qty from start of day till current hour over the last 4 months
- Multiply by P80 all-time-high quantity and P70 retailers

## Action Logic
- **On Track (±10%)**: No action
- **Growing (>110%)**: Deactivate discounts or increase price, reduce cart if too open
- **Dropping (<90%)**: Reduce price, increase cart by 20%
- **Zero Demand (qty=0 today)**: Market min + SKU discount


In [1]:
# =============================================================================
# IMPORTS AND SETUP
# =============================================================================
import pandas as pd
import numpy as np
import os
from datetime import datetime
import pytz
import sys
sys.path.append('..')

# Run queries_module - this:
# 1. Initializes Snowflake credentials (setup_environment_2.initialize_env())
# 2. Provides query_snowflake() function
# 3. Provides TIMEZONE from Snowflake
# 4. Provides get_current_stocks(), get_current_prices(), get_current_wac(), get_current_cart_rules()
%run queries_module.ipynb

# Run market_data_module - this:
# 1. Provides get_market_data() for fetching fresh market prices (NO INPUT REQUIRED)
# 2. Provides get_margin_tiers() for fetching margin tiers (NO INPUT REQUIRED)
# 3. Fetches Ben Soliman, Marketplace, and Scrapped prices
# 4. Fills missing prices from group-level data
# 5. Calculates market price percentiles and margin tiers
%run market_data_module_2.ipynb

# Cairo timezone
CAIRO_TZ = pytz.timezone('Africa/Cairo')
CAIRO_NOW = datetime.now(CAIRO_TZ)
TODAY = CAIRO_NOW.date()
CURRENT_HOUR = CAIRO_NOW.hour

# Configuration
UTH_GROWING_THRESHOLD = 1.10    # >110% = Growing (used for discount decisions)
UTH_DROPPING_THRESHOLD = 0.90   # <90% = Dropping (used for discount decisions)

# Stricter thresholds for actual price changes (discounts still use the old thresholds above)
QTY_PRICE_INCREASE_THRESHOLD = 1.2       # qty_ratio must exceed this to increase price
QTY_PRICE_DECREASE_THRESHOLD = 0.8       # qty_ratio must be below this to decrease price
RETAILER_PRICE_INCREASE_THRESHOLD = 1.20  # retailer_ratio must exceed this to increase price
RETAILER_PRICE_DECREASE_THRESHOLD = 0.80  # retailer_ratio must be below this to decrease price

LOW_STOCK_DOH_THRESHOLD = 1     # SKUs with DOH <= this are protected from price reduction
CART_INCREASE_PCT = 0.25        # 20% cart increase
CART_DECREASE_PCT = 0.25        # 20% cart decrease
MIN_CART_RULE = 10
MAX_CART_RULE = 300
MIN_PRICE_CHANGE_EGP = 0.5     # Minimum 0.25 EGP for any price change
MAX_PRICE_REDUCTIONS_PER_DAY = 3  # Max price reductions per day
# SKU discount percentage will be decided in sku_discount_handler

# Input/Output configuration
# Data is now loaded from Snowflake instead of Excel
INPUT_TABLE = 'MATERIALIZED_VIEWS.Pricing_data_extraction'
PREVIOUS_OUTPUT_TABLE = 'MATERIALIZED_VIEWS.pricing_periodic_push'
OUTPUT_FILE = f'module_3_output_{CAIRO_NOW.strftime("%Y%m%d_%H%M")}.xlsx'

print(f"Module 3: Periodic Actions")
print(f"Run Time (Cairo): {CAIRO_NOW.strftime('%Y-%m-%d %H:%M:%S')}")
print(f"Current Hour (Cairo): {CURRENT_HOUR}")
print(f"Input: {INPUT_TABLE} (today's data)")


/home/ec2-user/.Renviron


/home/ec2-user/service_account_key.json


/home/ec2-user/anaconda3/envs/python3/lib/python3.12/site-packages/snowflake/connector/options.py:104: UserWarning: You have an incompatible version of 'pyarrow' installed (20.0.0), please install a version that adheres to: 'pyarrow<19.0.0; extra == "pandas"'
  warn_incompatible_dep(


Queries Module | Timezone: America/Los_Angeles
✅ UTH and Last Hour functions defined
✅ Yesterday closing stock function defined
Fixed price & cart rule functions defined
get_commercial_min_prices() defined
get_commercial_price_ups() defined
get_margin_boundaries_region() defined
get_margin_boundaries_global() defined

QUERIES MODULE READY

Live Data Functions:
  • get_current_stocks()
  • get_packing_units()
  • get_current_prices()
  • get_current_wac()
  • get_current_cart_rules()

UTH Performance Functions:
  • get_uth_performance()         - UTH qty/retailers (Snowflake)
  • get_hourly_distribution()     - Historical hour contributions (Snowflake)
  • get_last_hour_performance()   - Last hour qty/retailers (DWH)

Stock Functions:
  • get_yesterday_closing_stock() - Yesterday closing stock with parent WH mapping

Override Functions:
  • get_fixed_prices()            - Fixed prices from Google Sheet

Commercial Constraints:
  • get_commercial_min_prices()   - Fresh min price constrai

/home/ec2-user/.Renviron
/home/ec2-user/service_account_key.json


Market Data Module loaded at 2026-04-18 23:03:24 Cairo time
Snowflake timezone: America/Los_Angeles
All queries defined ✓
Helper functions defined ✓
get_market_data() function defined ✓
get_margin_tiers() function defined ✓
get_brand_market_percentiles() function defined ✓
fill_brand_market_fallback() function defined ✓

MARKET DATA MODULE READY

Available functions (NO INPUT REQUIRED):
  - get_market_data()              : Fetch and process all market prices
  - get_margin_tiers()             : Fetch and calculate margin tiers
  - get_brand_market_percentiles() : Fetch brand-level market margin percentiles
  - fill_brand_market_fallback()   : Fill NaN market cols with brand percentiles

Usage:
  %run market_data_module.ipynb
  df_market = get_market_data()
  df_tiers = get_margin_tiers()
  df_brand_percs = get_brand_market_percentiles()
  df = fill_brand_market_fallback(df, df_brand_percs)
get_market_signals() function defined


/home/ec2-user/.Renviron
/home/ec2-user/service_account_key.json


Queries Module | Timezone: America/Los_Angeles
✅ UTH and Last Hour functions defined
✅ Yesterday closing stock function defined
Fixed price & cart rule functions defined
get_commercial_min_prices() defined
get_commercial_price_ups() defined
get_margin_boundaries_region() defined
get_margin_boundaries_global() defined

QUERIES MODULE READY

Live Data Functions:
  • get_current_stocks()
  • get_packing_units()
  • get_current_prices()
  • get_current_wac()
  • get_current_cart_rules()

UTH Performance Functions:
  • get_uth_performance()         - UTH qty/retailers (Snowflake)
  • get_hourly_distribution()     - Historical hour contributions (Snowflake)
  • get_last_hour_performance()   - Last hour qty/retailers (DWH)

Stock Functions:
  • get_yesterday_closing_stock() - Yesterday closing stock with parent WH mapping

Override Functions:
  • get_fixed_prices()            - Fixed prices from Google Sheet

Commercial Constraints:
  • get_commercial_min_prices()   - Fresh min price constrai

In [2]:
# =============================================================================
# LOAD PREVIOUS ACTIONS (Track price reductions per day)
# Now loads from Snowflake instead of local Excel files
# =============================================================================

def load_previous_actions():
    """Load previous Module 3 outputs from today (from Snowflake) to track price reductions."""
    try:
        # Query today's previous actions from Snowflake
        query = f"""
        SELECT * FROM {PREVIOUS_OUTPUT_TABLE}
        WHERE DATE(created_at) = '{TODAY}'
        ORDER BY created_at
        """
        df = query_snowflake(query)
        
        if len(df) == 0:
            print("No previous Module 3 outputs found for today. This is the first run.")
            return pd.DataFrame()
        
        print(f"Loaded {len(df)} previous action records from Snowflake")
        return df
    except Exception as e:
        print(f"Error loading previous actions from Snowflake: {e}")
        print("This may be the first run or table doesn't exist yet.")
        return pd.DataFrame()

def count_price_increase_today(product_id, warehouse_id, previous_df):
    """Count how many price increase this SKU has had today."""
    if previous_df.empty:
        return 0
    
    mask = (
        (previous_df['product_id'] == product_id) & 
        (previous_df['warehouse_id'] == warehouse_id) &
        (previous_df['price_action'].str.contains('increase', na=False))
    )
    return mask.sum()
    

print("Loading previous actions from today...")
df_previous_actions = load_previous_actions()
print(f"Previous actions loaded: {len(df_previous_actions)} records")


Loading previous actions from today...


Loaded 59582 previous action records from Snowflake
Previous actions loaded: 59582 records


In [3]:
try:
    prev_inc = (
        df_previous_actions.assign(
            inc_flag=df_previous_actions['price_action'].str.contains('increase', case=False, na=False)
        )
        .groupby(['product_id', 'warehouse_id'])['inc_flag']
        .sum()
        .reset_index(name='increase_count')
    )
except:
    prev_inc = pd.DataFrame(columns=['product_id', 'warehouse_id','increase_count'])
try:    
    prev_red = (
    df_previous_actions.assign(
        red_flag=df_previous_actions['price_action'].str.contains('decrease', case=False, na=False)
    )
    .groupby(['product_id', 'warehouse_id'])['red_flag']
    .sum()
    .reset_index(name='reduced_count') 
    )
except:
    prev_red = pd.DataFrame(columns=['product_id', 'warehouse_id','reduced_count'])

In [4]:
# =============================================================================
# LOAD MODULE 4 INCREASES TODAY (Cross-module synchronization)
# =============================================================================
# Prevent double price increases: count Module 4's performance-based increases
# toward Module 3's daily increase cap so the combined total is respected.
print("Loading Module 4 price increases from today...")

def load_module4_increases_today():
    """Load Module 4 performance-based increase counts per SKU today."""
    try:
        query = """
        SELECT product_id, warehouse_id, COUNT(*) as m4_increase_count
        FROM MATERIALIZED_VIEWS.pricing_hourly_push
        WHERE DATE(created_at) = CURRENT_DATE
          AND price_action IN ('rets_growing', 'qty_growing_price_step', 'above_market_surge')
        GROUP BY product_id, warehouse_id
        """
        df = query_snowflake(query)
        return df
    except Exception as e:
        print(f"  Note: Could not load Module 4 increases: {e}")
        return pd.DataFrame(columns=['product_id', 'warehouse_id', 'm4_increase_count'])

df_m4_increases = load_module4_increases_today()
print(f"  SKUs increased by Module 4 today: {len(df_m4_increases)}")
if len(df_m4_increases) > 0:
    print(f"  Total Module 4 increase actions today: {df_m4_increases['m4_increase_count'].sum()}")

# Merge Module 4 increase counts into prev_inc for unified daily cap
if len(df_m4_increases) > 0:
    prev_inc = prev_inc.merge(df_m4_increases, on=['product_id', 'warehouse_id'], how='outer')
    prev_inc['increase_count'] = prev_inc['increase_count'].fillna(0).astype(int)
    prev_inc['m4_increase_count'] = prev_inc['m4_increase_count'].fillna(0).astype(int)
else:
    prev_inc['m4_increase_count'] = 0
print(f"  Combined increase tracking ready (Module 3 + Module 4)")

Loading Module 4 price increases from today...


  SKUs increased by Module 4 today: 838
  Total Module 4 increase actions today: 1063
  Combined increase tracking ready (Module 3 + Module 4)


In [5]:
# =============================================================================
# SNOWFLAKE CONNECTION
# =============================================================================
# query_snowflake() and TIMEZONE are provided by queries_module.ipynb
# (which also initializes Snowflake credentials from setup_environment_2)
print(f"Snowflake connection ready")
print(f"Timezone: {TIMEZONE}")


Snowflake connection ready
Timezone: America/Los_Angeles


In [6]:
# =============================================================================
# QUERY 1: TODAY'S UTH PERFORMANCE
# =============================================================================
UTH_LIVE_QUERY = f'''
WITH parent_whs AS (
    SELECT * FROM (VALUES (236,343),(1,467),(962,343)) x(parent_id,child_id)
),
params AS (
    SELECT
        CONVERT_TIMEZONE('{TIMEZONE}', 'Africa/Cairo', CURRENT_TIMESTAMP())::DATE AS today,
        HOUR(CONVERT_TIMEZONE('{TIMEZONE}', 'Africa/Cairo', CURRENT_TIMESTAMP())) AS current_hour
),

-- Map dynamic tags to warehouse IDs using name matching
qd_det AS (
    SELECT DISTINCT 
        dt.id AS tag_id, 
        dt.name AS tag_name,
        REPLACE(w.name, ' ', '') AS warehouse_name,
        w.id AS warehouse_id,
        warehouse_name ILIKE '%' || CASE 
            WHEN SPLIT_PART(tag_name, '_', 1) = 'El' THEN SPLIT_PART(tag_name, '_', 2) 
            ELSE SPLIT_PART(tag_name, '_', 1) 
        END || '%' AS contains_flag
    FROM dynamic_tags dt
    JOIN dynamic_taggables dta ON dt.id = dta.dynamic_tag_id 
    CROSS JOIN warehouses w 
    WHERE dt.id > 3000
        AND dt.name LIKE '%QD_rets%'
        AND w.id IN (1, 236, 337, 8, 339, 170, 501, 401, 703, 632, 797, 962)
        AND contains_flag = 'true'
),

-- Get current active QD configurations
qd_config AS (
    SELECT * 
    FROM (
        SELECT 
            product_id,
            start_at,
            end_at,
            packing_unit_id,
            id AS qd_id,
            qd.warehouse_id,
            MAX(CASE WHEN tier = 1 THEN quantity END) AS tier_1_qty,
            MAX(CASE WHEN tier = 1 THEN discount_percentage END) AS tier_1_discount_pct,
            MAX(CASE WHEN tier = 2 THEN quantity END) AS tier_2_qty,
            MAX(CASE WHEN tier = 2 THEN discount_percentage END) AS tier_2_discount_pct,
            MAX(CASE WHEN tier = 3 THEN quantity END) AS tier_3_qty,
            MAX(CASE WHEN tier = 3 THEN discount_percentage END) AS tier_3_discount_pct
        FROM (
            SELECT 
                qd.id,
                qdv.product_id,
                qdv.packing_unit_id,
                qdv.quantity,
                qdv.discount_percentage,
                qd.dynamic_tag_id,
                qd.start_at,
                qd.end_at,
                ROW_NUMBER() OVER (
                    PARTITION BY qdv.product_id, qdv.packing_unit_id, qd.id 
                    ORDER BY qdv.quantity
                ) AS tier
            FROM quantity_discounts qd 
            JOIN quantity_discount_values qdv ON qdv.quantity_discount_id = qd.id
            WHERE active = 'true'
        ) qd_tiers
        JOIN qd_det qd ON qd.tag_id = qd_tiers.dynamic_tag_id
        GROUP BY ALL
    )
    QUALIFY ROW_NUMBER() OVER (PARTITION BY product_id, packing_unit_id, warehouse_id ORDER BY start_at DESC) = 1
),

-- Today's sales up-till-hour with discount breakdown
today_uth_sales AS (
    SELECT 
        COALESCE(pwh.parent_id, pso.warehouse_id) AS warehouse_id,
        pso.product_id,
        so.retailer_id,
        pso.packing_unit_id,
        pso.purchased_item_count AS qty,
        pso.total_price AS nmv,
        pso.ITEM_DISCOUNT_VALUE AS sku_discount_per_unit,
        pso.ITEM_QUANTITY_DISCOUNT_VALUE AS qty_discount_per_unit,
        qd.tier_1_qty,
        qd.tier_2_qty,
        qd.tier_3_qty,
        -- Determine tier used
        CASE 
            WHEN pso.ITEM_QUANTITY_DISCOUNT_VALUE = 0 OR qd.tier_1_qty IS NULL THEN 'Base'
            WHEN qd.tier_3_qty IS NOT NULL AND pso.purchased_item_count >= qd.tier_3_qty THEN 'Tier 3'
            WHEN qd.tier_2_qty IS NOT NULL AND pso.purchased_item_count >= qd.tier_2_qty THEN 'Tier 2'
            WHEN qd.tier_1_qty IS NOT NULL AND pso.purchased_item_count >= qd.tier_1_qty THEN 'Tier 1'
            ELSE 'Base'
        END AS tier_used
    FROM product_sales_order pso
    LEFT JOIN parent_whs pwh ON pwh.child_id = pso.warehouse_id
    JOIN sales_orders so ON so.id = pso.sales_order_id
    LEFT JOIN qd_config qd 
        ON qd.product_id = pso.product_id 
        AND qd.packing_unit_id = pso.packing_unit_id
        AND qd.warehouse_id = COALESCE(pwh.parent_id, pso.warehouse_id)
    CROSS JOIN params p
    WHERE so.created_at::DATE = p.today
        AND HOUR(so.created_at) < p.current_hour
        AND so.sales_order_status_id NOT IN (7, 12)
        AND so.channel IN ('telesales', 'retailer')
        AND pso.purchased_item_count <> 0
)

SELECT 
    warehouse_id,
    product_id,
    SUM(qty) AS uth_qty,
    SUM(nmv) AS uth_nmv,
    COUNT(DISTINCT retailer_id) AS uth_retailers,
    -- SKU discount NMV and contribution
    SUM(CASE WHEN sku_discount_per_unit > 0 THEN nmv ELSE 0 END) AS sku_discount_nmv_uth,
    ROUND(SUM(CASE WHEN sku_discount_per_unit > 0 THEN nmv ELSE 0 END) * 100.0 / NULLIF(SUM(nmv), 0), 2) AS sku_disc_cntrb_uth,
    -- Quantity discount NMV and contribution
    SUM(CASE WHEN qty_discount_per_unit > 0 THEN nmv ELSE 0 END) AS qty_discount_nmv_uth,
    ROUND(SUM(CASE WHEN qty_discount_per_unit > 0 THEN nmv ELSE 0 END) * 100.0 / NULLIF(SUM(nmv), 0), 2) AS qty_disc_cntrb_uth,
    -- Tier-level NMV
    SUM(CASE WHEN tier_used = 'Tier 1' THEN nmv ELSE 0 END) AS t1_nmv_uth,
    SUM(CASE WHEN tier_used = 'Tier 2' THEN nmv ELSE 0 END) AS t2_nmv_uth,
    SUM(CASE WHEN tier_used = 'Tier 3' THEN nmv ELSE 0 END) AS t3_nmv_uth,
    -- Tier-level contributions
    ROUND(SUM(CASE WHEN tier_used = 'Tier 1' THEN nmv ELSE 0 END) * 100.0 / NULLIF(SUM(nmv), 0), 2) AS t1_cntrb_uth,
    ROUND(SUM(CASE WHEN tier_used = 'Tier 2' THEN nmv ELSE 0 END) * 100.0 / NULLIF(SUM(nmv), 0), 2) AS t2_cntrb_uth,
    ROUND(SUM(CASE WHEN tier_used = 'Tier 3' THEN nmv ELSE 0 END) * 100.0 / NULLIF(SUM(nmv), 0), 2) AS t3_cntrb_uth
FROM today_uth_sales
GROUP BY warehouse_id, product_id
HAVING SUM(nmv) > 0
'''

print("Loading today's UTH performance with discount contributions...")
df_uth_today = query_snowflake(UTH_LIVE_QUERY)
print(f"Loaded {len(df_uth_today)} UTH records")


Loading today's UTH performance with discount contributions...


Loaded 9687 UTH records


In [7]:
# =============================================================================
# QUERY 2: HISTORICAL HOURLY DISTRIBUTION (Last 4 Months) - By Category & Warehouse
# =============================================================================
# Uses get_hourly_distribution() from queries_module

df_hourly_dist = get_hourly_distribution()

# Rename column for backwards compatibility with rest of Module 3
df_hourly_dist['avg_uth_pct'] = df_hourly_dist['avg_uth_pct_qty']
print(f"Using avg_uth_pct_qty as avg_uth_pct for Module 3 compatibility")

# Per-hour contribution (0..23) by warehouse + cat for in-stock hours weighting
df_hourly_curve = get_hourly_contribution_by_hour()

# Today's hourly stock snapshots for in-stock hours
df_stock_snapshots = get_stock_snapshots_today()


Fetching hourly distribution from Snowflake...


  Loaded 791 hourly distribution records
Using avg_uth_pct_qty as avg_uth_pct for Module 3 compatibility
Fetching hourly contribution by hour from Snowflake...


  Loaded 18033 hourly contribution records
Fetching today's stock snapshots from Snowflake...


  Loaded 437265 stock snapshot rows


In [8]:
# =============================================================================
# QUERY 3 & 4: ACTIVE DISCOUNTS
# =============================================================================

# SKU Discounts query (from data_extraction.ipynb)
ACTIVE_SKU_DISCOUNTS_QUERY = f'''
WITH parent_whs AS (
    SELECT * FROM (VALUES (236,343),(1,467),(962,343)) x(parent_id,child_id)
),
active_sku_discount AS ( 
    SELECT 
        x.id AS sku_discount_id,
        retailer_id,
        product_id,
        packing_unit_id,
        DISCOUNT_PERCENTAGE,
        start_at,
        end_at 
    FROM (
        SELECT 
            sd.*,
            f.value::INT AS retailer_id 
        FROM SKU_DISCOUNTS sd,
        LATERAL FLATTEN(
            input => SPLIT(
                REPLACE(REPLACE(REPLACE(sd.retailer_ids, '{{', ''), '}}', ''), '"', ''), 
                ','
            )
        ) f
        WHERE start_at::DATE <= CONVERT_TIMEZONE('{TIMEZONE}', 'Africa/Cairo', CURRENT_TIMESTAMP())::DATE
        and end_at::DATE >= CONVERT_TIMEZONE('{TIMEZONE}', 'Africa/Cairo', CURRENT_TIMESTAMP())::DATE
            AND active = 'true'
    ) x 
    JOIN SKU_DISCOUNT_VALUES sdv ON x.id = sdv.sku_discount_id
    WHERE name_en = 'Special Discounts'
    QUALIFY MAX(start_at) OVER (PARTITION BY retailer_id, product_id, packing_unit_id) = start_at 
)

SELECT 
    product_id, 
    COALESCE(pwh.parent_id, sub.warehouse_id) AS warehouse_id,
    AVG(DISCOUNT_PERCENTAGE) AS active_sku_disc_pct,
    1 AS has_active_sku_discount
FROM (
    SELECT 
        asd.*,
        warehouse_id 
    FROM active_sku_discount asd 
    JOIN materialized_views.retailer_polygon rp ON rp.retailer_id = asd.retailer_id
    JOIN WAREHOUSE_DISPATCHING_RULES wdr ON wdr.product_id = asd.product_id
    JOIN DISPATCHING_POLYGONS dp ON dp.id = wdr.DISPATCHING_POLYGON_ID AND dp.district_id = rp.district_id
) sub
LEFT JOIN parent_whs pwh ON pwh.child_id = sub.warehouse_id
GROUP BY ALL
'''

# Active QD Query - Reuses the same CTE structure from UTH_LIVE_QUERY
ACTIVE_QD_QUERY = f'''
WITH qd_det AS (
    SELECT DISTINCT 
        dt.id AS tag_id, 
        dt.name AS tag_name,
        REPLACE(w.name, ' ', '') AS warehouse_name,
        w.id AS warehouse_id,
        warehouse_name ILIKE '%' || CASE 
            WHEN SPLIT_PART(tag_name, '_', 1) = 'El' THEN SPLIT_PART(tag_name, '_', 2) 
            ELSE SPLIT_PART(tag_name, '_', 1) 
        END || '%' AS contains_flag
    FROM dynamic_tags dt
    JOIN dynamic_taggables dta ON dt.id = dta.dynamic_tag_id 
    CROSS JOIN warehouses w 
    WHERE dt.id > 3000
        AND dt.name LIKE '%QD_rets%'
        AND w.id IN (1, 236, 337, 8, 339, 170, 501, 401, 703, 632, 797, 962)
        AND contains_flag = 'true'
),

qd_config AS (
    SELECT * 
    FROM (
        SELECT 
            product_id,
            packing_unit_id,
            qd.warehouse_id,
            MAX(CASE WHEN tier = 1 THEN quantity END) AS qd_tier_1_qty,
            MAX(CASE WHEN tier = 1 THEN discount_percentage END) AS qd_tier_1_disc_pct,
            MAX(CASE WHEN tier = 2 THEN quantity END) AS qd_tier_2_qty,
            MAX(CASE WHEN tier = 2 THEN discount_percentage END) AS qd_tier_2_disc_pct,
            MAX(CASE WHEN tier = 3 THEN quantity END) AS qd_tier_3_qty,
            MAX(CASE WHEN tier = 3 THEN discount_percentage END) AS qd_tier_3_disc_pct
        FROM (
            SELECT 
                qd.id,
                qdv.product_id,
                qdv.packing_unit_id,
                qdv.quantity,
                qdv.discount_percentage,
                qd.dynamic_tag_id,
                qd.start_at,
                ROW_NUMBER() OVER (
                    PARTITION BY qdv.product_id, qdv.packing_unit_id, qd.id 
                    ORDER BY qdv.quantity
                ) AS tier
            FROM quantity_discounts qd 
            JOIN quantity_discount_values qdv ON qdv.quantity_discount_id = qd.id
            WHERE  active = TRUE
        ) qd_tiers
        JOIN qd_det qd ON qd.tag_id = qd_tiers.dynamic_tag_id
        GROUP BY ALL
    )
    QUALIFY ROW_NUMBER() OVER (PARTITION BY product_id, packing_unit_id, warehouse_id ORDER BY qd_tier_1_qty DESC) = 1
)

SELECT 
    product_id,
    warehouse_id,
    qd_tier_1_qty,
    qd_tier_1_disc_pct,
    qd_tier_2_qty,
    qd_tier_2_disc_pct,
    qd_tier_3_qty,
    qd_tier_3_disc_pct,
    1 AS has_active_qd
FROM qd_config
'''

print("Loading active SKU discounts...")
df_active_sku_disc = query_snowflake(ACTIVE_SKU_DISCOUNTS_QUERY)
print(f"Loaded {len(df_active_sku_disc)} active SKU discount records")

print("Loading active Quantity discounts...")
df_active_qd = query_snowflake(ACTIVE_QD_QUERY)
print(f"Loaded {len(df_active_qd)} active QD records")


Loading active SKU discounts...


Loaded 7951 active SKU discount records
Loading active Quantity discounts...


Loaded 1864 active QD records


In [9]:
# =============================================================================
# LOAD DATA FROM SNOWFLAKE (Instead of Excel file)
# =============================================================================
print("Loading data from Snowflake...")

# Query to get today's data from Pricing_data_extraction
LOAD_QUERY = f"""
SELECT * FROM {INPUT_TABLE}
WHERE created_at = '{datetime.now(CAIRO_TZ).date()}'
"""

df = query_snowflake(LOAD_QUERY)
print(f"Loaded {len(df)} records from Snowflake")

# Refresh live data using queries_module
print("\nRefreshing live data...")

# Refresh stocks
df_fresh_stocks = get_current_stocks()
df = df.drop(columns=['stocks'], errors='ignore')
df = df.merge(df_fresh_stocks, on=['warehouse_id', 'product_id'], how='left')
df['stocks'] = df['stocks'].fillna(0)

# Refresh current prices
df_fresh_prices = get_current_prices()
df = df.drop(columns=['current_price'], errors='ignore')
df = df.merge(df_fresh_prices[['cohort_id', 'product_id', 'current_price']], 
              on=['cohort_id', 'product_id'], how='left')

# Refresh WAC
df_fresh_wac = get_current_wac()
df = df.drop(columns=['wac_p'], errors='ignore')
df = df.merge(df_fresh_wac, on='product_id', how='left')

# Refresh cart rules
df_fresh_cart = get_current_cart_rules()
df = df.drop(columns=['current_cart_rule'], errors='ignore')
df = df.merge(df_fresh_cart, on=['cohort_id', 'product_id'], how='left')

print(f"Live data refreshed: stocks, prices, WAC, cart rules")

# Refresh yesterday's closing stock (for zero demand validation)
df_closing_stock = get_yesterday_closing_stock()
df = df.drop(columns=['closing_stock_yesterday'], errors='ignore')
df = df.merge(df_closing_stock, on=['warehouse_id', 'product_id'], how='left')
df['closing_stock_yesterday'] = df['closing_stock_yesterday'].fillna(0)
print(f"  Yesterday closing stock merged: {(df['closing_stock_yesterday'] > 0).sum()} SKUs had stock at close")

# =============================================================================
# =============================================================================
# LOAD PERCENTILE DATA FOR CART RULES
# =============================================================================
df_percentiles = get_percentile_data()

# Refresh market prices and margin tiers using new standalone functions
print("\nRefreshing market prices and margin tiers...")

# Get fresh market data (no input required)
df_fresh_market = get_market_data()
print(f"  Fetched {len(df_fresh_market)} market data records")

# Get fresh margin tiers (no input required)
df_fresh_tiers = get_margin_tiers()
print(f"  Fetched {len(df_fresh_tiers)} margin tier records")

# Drop old market columns and merge fresh data
market_cols_to_drop = [
    'below_market', 'market_min', 'market_25', 'market_50', 
    'market_75', 'market_max', 'above_market',
    'minimum', 'percentile_25', 'percentile_50', 'percentile_75', 'maximum',
    'ben_soliman_price', 'final_min_price', 'final_max_price', 'final_mod_price',
    'final_true_min', 'final_true_max', 'min_scrapped', 'scrapped25', 
    'scrapped50', 'scrapped75', 'max_scrapped',
    'market_data_source'
]
df = df.drop(columns=[c for c in market_cols_to_drop if c in df.columns], errors='ignore')

# Merge fresh market data
df = df.merge(
    df_fresh_market, 
    on=['cohort_id', 'product_id','region'], 
    how='left'
)

# Drop old margin tier columns and merge fresh data
margin_tier_cols_to_drop = [
    'margin_tier_below', 'margin_tier_1', 'margin_tier_2', 'margin_tier_3',
    'margin_tier_4', 'margin_tier_5', 'margin_tier_above_1', 'margin_tier_above_2',
    'optimal_bm', 'min_boundary', 'max_boundary', 'median_bm',
    'effective_min_margin', 'margin_step'
]
df = df.drop(columns=[c for c in margin_tier_cols_to_drop if c in df.columns], errors='ignore')

# Merge fresh margin tiers (by warehouse_id + product_id)
margin_tier_merge_keys = ['warehouse_id', 'product_id']
df = df.drop(columns=[c for c in df_fresh_tiers.columns if c in df.columns and c not in margin_tier_merge_keys], errors='ignore')
df = df.merge(
    df_fresh_tiers, 
    on=margin_tier_merge_keys, 
    how='left'
)

print(f"Market data refreshed")

# Apply brand market percentile fallback for SKUs without market data
print("\nApplying brand market percentile fallback...")
df_brand_percs = get_brand_market_percentiles()
df = fill_brand_market_fallback(df, df_brand_percs)
print(f"  Market data source distribution: {df['market_data_source'].value_counts(dropna=False).to_dict()}")

# V2 price tiers for pricing decisions
print("\nLoading V2 price tiers...")
df_market_v2 = get_market_data_v2()
df_market_cohorts = expand_to_cohorts(df_market_v2)
df = df.merge(
    df_market_cohorts[['product_id', 'cohort_id', 'price_tiers']],
    on=['product_id', 'cohort_id'], how='left'
)
df['price_tiers'] = df['price_tiers'].apply(lambda x: x if isinstance(x, list) else [])

# Build margin_tier_prices from margin tier columns
margin_tier_cols = ['margin_tier_below', 'margin_tier_1', 'margin_tier_2', 'margin_tier_3',
                    'margin_tier_4', 'margin_tier_5', 'margin_tier_above_1', 'margin_tier_above_2']

def build_margin_tier_prices(row):
    wac = row.get('wac_p', 0)
    if wac <= 0:
        return []
    prices = []
    for col in margin_tier_cols:
        m = row.get(col)
        if pd.notna(m) and 0 < m < 1:
            prices.append(round(wac / (1 - m) * 4) / 4)
    return sorted(set(prices))

df['margin_tier_prices'] = df.apply(build_margin_tier_prices, axis=1)

def get_effective_tiers(row):
    if row['price_tiers'] and len(row['price_tiers']) > 0:
        return row['price_tiers']
    if row['margin_tier_prices'] and len(row['margin_tier_prices']) > 0:
        return row['margin_tier_prices']
    return []

df['effective_tiers'] = df.apply(get_effective_tiers, axis=1)
print(f"  V2 price tiers: {(df['price_tiers'].apply(len) > 0).sum()} SKUs")
print(f"  Effective tiers: {(df['effective_tiers'].apply(len) > 0).sum()} SKUs")

# Refresh commercial min price constraints (fresh from finance.minimum_prices)
print("\nRefreshing commercial min prices...")
df_fresh_commercial = get_commercial_min_prices()
df = df.drop(columns=['commercial_min_price'], errors='ignore')
df = df.merge(df_fresh_commercial, on=['product_id', 'region'], how='left')
df['commercial_min_price'] = df['commercial_min_price'].fillna(0)
print(f"  {(df['commercial_min_price'] > 0).sum()} SKUs with commercial min price")

# Merge UTH today data - drop old columns first
uth_cols = ['uth_qty', 'uth_nmv', 'uth_retailers', 'sku_discount_nmv_uth', 'sku_disc_cntrb_uth',
            'qty_discount_nmv_uth', 'qty_disc_cntrb_uth', 't1_nmv_uth', 't2_nmv_uth', 't3_nmv_uth',
            't1_cntrb_uth', 't2_cntrb_uth', 't3_cntrb_uth']
df = df.drop(columns=[c for c in uth_cols if c in df.columns], errors='ignore')

if len(df_uth_today) > 0:
    df = df.merge(df_uth_today, on=['warehouse_id', 'product_id'], how='left')
else:
    for col in uth_cols:
        df[col] = 0

# Merge hourly distribution - drop old column first (now by warehouse_id + cat)
df = df.drop(columns=['avg_uth_pct'], errors='ignore')
if len(df_hourly_dist) > 0:
    df = df.merge(df_hourly_dist, on=['warehouse_id', 'cat'], how='left')
else:
    df['avg_uth_pct'] = 0.5  # Default 50%

# In-stock hours contribution: sum of (cat, warehouse) hour contribution only for hours when SKU had stock
df = df.drop(columns=['in_stock_contribution_qty', 'in_stock_contribution_ret'], errors='ignore')
if len(df_stock_snapshots) > 0 and len(df_hourly_curve) > 0:
    in_stock = df_stock_snapshots[(df_stock_snapshots['available_stock'] > 0) & (df_stock_snapshots['hour'] < CURRENT_HOUR)][['product_id', 'warehouse_id', 'hour','cat']].drop_duplicates()
    if len(in_stock) > 0:
        in_stock = in_stock.merge(df_hourly_curve, on=['warehouse_id', 'cat', 'hour'], how='left')
        contrib = in_stock.groupby(['product_id', 'warehouse_id'], as_index=False).agg(
            in_stock_contribution_qty=('pct_contribution_qty', 'sum'),
            in_stock_contribution_ret=('pct_contribution_retailers', 'sum'))
        df = df.merge(contrib, on=['product_id', 'warehouse_id'], how='left')
if 'in_stock_contribution_qty' not in df.columns:
    df['in_stock_contribution_qty'] = np.nan
if 'in_stock_contribution_ret' not in df.columns:
    df['in_stock_contribution_ret'] = np.nan
# No snapshots / OOS all hours / missing contribution -> full in stock (use avg_uth_pct)
df['in_stock_contribution_qty'] = df['in_stock_contribution_qty'].fillna(df['avg_uth_pct'])
df['in_stock_contribution_ret'] = df['in_stock_contribution_ret'].fillna(df['avg_uth_pct'])
# When contribution sum is 0 (OOS all hours with snapshots), treat as full in stock
df.loc[df['in_stock_contribution_qty'] <= 0, 'in_stock_contribution_qty'] = df['avg_uth_pct']
df.loc[df['in_stock_contribution_ret'] <= 0, 'in_stock_contribution_ret'] = df['avg_uth_pct']

# Merge active SKU discounts - drop old columns first
sku_disc_cols = ['has_active_sku_discount', 'active_sku_disc_pct', 'active_sku_discount_value']
df = df.drop(columns=[c for c in sku_disc_cols if c in df.columns], errors='ignore')

if len(df_active_sku_disc) > 0:
    df = df.merge(df_active_sku_disc, on=['warehouse_id', 'product_id'], how='left')
else:
    df['has_active_sku_discount'] = 0
    df['active_sku_disc_pct'] = 0

# Merge active QD - drop old columns first
qd_cols = ['has_active_qd', 'qd_tier_1_qty', 'qd_tier_1_disc_pct', 
           'qd_tier_2_qty', 'qd_tier_2_disc_pct', 'qd_tier_3_qty', 'qd_tier_3_disc_pct']
df = df.drop(columns=[c for c in qd_cols if c in df.columns], errors='ignore')

if len(df_active_qd) > 0:
    df = df.merge(df_active_qd, on=['warehouse_id', 'product_id'], how='left')
else:
    df['has_active_qd'] = 0
    df['qd_tier_1_qty'] = 0
    df['qd_tier_1_disc_pct'] = 0
    df['qd_tier_2_qty'] = 0
    df['qd_tier_2_disc_pct'] = 0
    df['qd_tier_3_qty'] = 0
    df['qd_tier_3_disc_pct'] = 0

# Fill NaN
df['uth_qty'] = df['uth_qty'].fillna(0)
df['uth_retailers'] = df['uth_retailers'].fillna(0)
df['avg_uth_pct'] = df['avg_uth_pct'].fillna(0.5)
df['has_active_sku_discount'] = df['has_active_sku_discount'].fillna(0)
df['active_sku_discount_value'] = df.get('active_sku_discount_value', pd.Series([0]*len(df))).fillna(0)
df['has_active_qd'] = df['has_active_qd'].fillna(0)
df['qd_tier_1_qty'] = df['qd_tier_1_qty'].fillna(0)
df['qd_tier_1_disc_pct'] = df['qd_tier_1_disc_pct'].fillna(0)
df['qd_tier_2_qty'] = df['qd_tier_2_qty'].fillna(0)
df['qd_tier_2_disc_pct'] = df['qd_tier_2_disc_pct'].fillna(0)
df['qd_tier_3_qty'] = df['qd_tier_3_qty'].fillna(0)
df['qd_tier_3_disc_pct'] = df['qd_tier_3_disc_pct'].fillna(0)

# Quarterly contribution factor for seasonal P80 adjustment
df_qtr_cntrb = get_quarterly_contribution()
df = df.merge(df_qtr_cntrb[['cat', 'qtr_cntrb']], on='cat', how='left')
df['qtr_cntrb'] = df['qtr_cntrb'].fillna(1.0)
print(f"  Quarterly contribution merged: min={df['qtr_cntrb'].min():.2f}, max={df['qtr_cntrb'].max():.2f}, mean={df['qtr_cntrb'].mean():.2f}")

# Target turnover qty for high-DOH SKUs
df_target_turnover = get_target_turnover_qty()
df = df.merge(df_target_turnover[['warehouse_id', 'product_id', 'target_qty']], on=['warehouse_id', 'product_id'], how='left')
print(f"  Target turnover merged: {df['target_qty'].notna().sum()} high-DOH SKUs have target_qty")

# =============================================================================
# TURNOVER-BASED DOH: Calculate responsive_doh using in_stock_rr (vectorized)
# =============================================================================
# responsive_doh = stocks / in_stock_rr (exponential-decay weighted running rate from data_extraction)
df['in_stock_rr'] = pd.to_numeric(df.get('in_stock_rr', 0), errors='coerce').fillna(0)
df['responsive_doh'] = np.where(
    df['in_stock_rr'] > 0,
    df['stocks'] / df['in_stock_rr'],
    999  # No running rate = infinite DOH
)

# min_induced_price = wac_p * (0.9 + target_margin * 0.5)
# This is the floor price for induced pricing when DOH > 30
df['target_margin'] = pd.to_numeric(df.get('target_margin', 0), errors='coerce').fillna(0)
df['min_induced_price'] = df['wac_p'] * (0.9)

print(f"Data merged. Total records: {len(df)}")
print(f"  SKUs with active SKU discount: {(df['has_active_sku_discount'] == 1).sum()}")
print(f"  SKUs with active QD: {(df['has_active_qd'] == 1).sum()}")
print(f"  SKUs with high DOH (>30): {(df['responsive_doh'] > 30).sum()}")


Loading data from Snowflake...


Loaded 29791 records from Snowflake

Refreshing live data...
Fetching current stocks...


  Loaded 1935170 records


Fetching current prices...


  Loaded 118669 records
Fetching current WAC...


...

  V2 price tiers: 24990 SKUs
  Effective tiers: 29792 SKUs

Refreshing commercial min prices...
Fetching commercial min prices...


  Loaded 767 commercial min price records
  1285 SKUs with commercial min price


  Fetching quarterly contribution factors...


    Found qtr_cntrb for 93 categories
  Quarterly contribution merged: min=0.90, max=1.10, mean=0.96
  Fetching target turnover quantities...


    Found target_qty for 13120 high-DOH SKUs
  Target turnover merged: 12212 high-DOH SKUs have target_qty
Data merged. Total records: 30161
  SKUs with active SKU discount: 7948
  SKUs with active QD: 1878
  SKUs with high DOH (>30): 6904


In [10]:
# =============================================================================
# HELPER FUNCTIONS
# =============================================================================

def find_next_price_above(current_price, row):
    """Find the first tier ABOVE current_price by at least MIN_PRICE_CHANGE_EGP.
    Uses effective_tiers (price_tiers > margin_tier_prices)."""
    current_price = float(current_price) if current_price else 0
    if pd.isna(current_price) or current_price <= 0:
        return current_price
    
    tiers = row.get('effective_tiers', [])
    if not tiers:
        return current_price
    
    for tier in tiers:
        if tier > current_price + MIN_PRICE_CHANGE_EGP:
            return round(tier, 2)
    
    return current_price

def _calc_avg_margin_step_m3(row):
    """Calculate average margin step from effective tiers."""
    wac = float(row.get('wac_p', 0) or 0)
    if wac <= 0:
        return None
    
    all_prices = sorted(set(p for p in row.get('effective_tiers', []) if p > wac))
    
    if len(all_prices) < 2:
        return None
    
    margins = [(p - wac) / p for p in all_prices]
    steps = [margins[i+1] - margins[i] for i in range(len(margins)-1)]
    
    if not steps:
        return None
    
    return np.mean(steps)

def find_next_price_below(current_price, row):
    """Find the first tier BELOW current_price by at least MIN_PRICE_CHANGE_EGP.
    Uses effective_tiers. When above all tiers, uses gradual step-down."""
    current_price = float(current_price) if current_price else 0
    if pd.isna(current_price) or current_price <= 0:
        return current_price
    
    tiers = row.get('effective_tiers', [])
    if not tiers:
        return current_price
    
    # Above all tiers — gradual step-down
    if current_price > tiers[-1] + MIN_PRICE_CHANGE_EGP:
        wac = float(row.get('wac_p', 0) or 0)
        if wac > 0 and current_price > wac:
            current_margin = (current_price - wac) / current_price
            
            avg_step = _calc_avg_margin_step_m3(row)
            if avg_step is not None:
                new_margin = current_margin - avg_step
                if new_margin > 0:
                    new_price = round(wac / (1 - new_margin) * 4) / 4
                    if new_price <= tiers[-1]:
                        return round(tiers[-1], 2)
                    return new_price
            
            target_margin = float(row.get('target_margin', 0) or 0)
            if target_margin > 0:
                new_margin = current_margin - target_margin * 0.20
                if new_margin > 0:
                    new_price = round(wac / (1 - new_margin) * 4) / 4
                    if new_price <= tiers[-1]:
                        return round(tiers[-1], 2)
                    return new_price
            
            new_price = round(current_price * 0.99 * 4) / 4
            if new_price <= tiers[-1]:
                return round(tiers[-1], 2)
            return new_price
    
    # Normal path — within tiers
    for tier in reversed(tiers):
        if tier < current_price - MIN_PRICE_CHANGE_EGP:
            return round(tier, 2)
    
    return current_price

def adjust_cart_rule(current_cart, direction, row):
    """Adjust cart rule by 20% up or down."""
    normal_refill = float(row.get('normal_refill', 5) or 5)
    stddev = float(row.get('refill_stddev', 2) or 2)
    current_cart = float(current_cart or 5)
    
    if direction == 'increase':
        new_cart = current_cart * (1 + CART_INCREASE_PCT)
        new_cart = min(new_cart, MAX_CART_RULE)
    else:  # decrease
        # Formula: max(0.8 * cart, normal_refill + 3*std)
        new_cart = current_cart * (1 - CART_DECREASE_PCT)
        min_floor = normal_refill + (3 * stddev)
        new_cart = max(new_cart, min_floor, MIN_CART_RULE)
    
    return int(new_cart)

print("Helper functions loaded.")


Helper functions loaded.


In [11]:
# =============================================================================
# PERCENTILE HELPER FUNCTIONS FOR CART RULES
# =============================================================================

def get_current_percentile_level(current_cart_rule, percentile_row):
    """Determine which percentile level current cart rule corresponds to."""
    if len(percentile_row) == 0:
        return None
    
    perc_95 = percentile_row.iloc[0]['perc_95']
    perc_75 = percentile_row.iloc[0]['perc_75']
    perc_50 = percentile_row.iloc[0]['perc_50']
    perc_25 = percentile_row.iloc[0]['perc_25']
    
    # Determine current level (with tolerance for rounding)
    if pd.notna(perc_95) and abs(current_cart_rule - perc_95) <= 1:
        return 95
    elif pd.notna(perc_75) and abs(current_cart_rule - perc_75) <= 1:
        return 75
    elif pd.notna(perc_50) and abs(current_cart_rule - perc_50) <= 1:
        return 50
    elif pd.notna(perc_25) and abs(current_cart_rule - perc_25) <= 1:
        return 25
    return None

def get_next_lower_percentile(current_level, percentile_row):
    """Get next lower percentile value."""
    if len(percentile_row) == 0:
        return None
    
    if current_level == 95:
        return percentile_row.iloc[0]['perc_75']
    elif current_level == 75:
        return percentile_row.iloc[0]['perc_50']
    elif current_level == 50:
        return percentile_row.iloc[0]['perc_25']
    elif current_level == 25:
        return percentile_row.iloc[0]['perc_25']  # Stay at minimum
    return None

print("Percentile helper functions loaded.")


Percentile helper functions loaded.


In [12]:
# =============================================================================
# HELPER: Calculate margin step from existing tier prices
# =============================================================================
def calculate_margin_step(row):
    """
    Calculate the margin step size from existing margin tiers.
    Used to induce prices below available tiers when DOH > 30.
    
    Returns:
        Average step size between consecutive tiers, or 0.25 * target_margin as default
    """
    target_margin = float(row.get('target_margin', 0.10) or 0.10)
    default_step = 0.25 * target_margin
    
    tier_cols = ['margin_tier_1', 'margin_tier_2', 'margin_tier_3', 
                 'margin_tier_4', 'margin_tier_5']
    tiers = [row.get(col) for col in tier_cols]
    valid_tiers = [t for t in tiers if pd.notna(t) and t is not None]
    
    if len(valid_tiers) >= 2:
        steps = [abs(valid_tiers[i+1] - valid_tiers[i]) for i in range(len(valid_tiers)-1)]
        return np.mean(steps) if steps else default_step
    
    # Fallback: use market margins if available
    market_cols = ['market_min', 'market_25', 'market_50', 'market_75', 'market_max']
    markets = [row.get(col) for col in market_cols]
    valid_markets = [m for m in markets if pd.notna(m) and m is not None]
    
    if len(valid_markets) >= 2:
        steps = [abs(valid_markets[i+1] - valid_markets[i]) for i in range(len(valid_markets)-1)]
        return np.mean(steps) if steps else default_step
    
    return default_step

def calculate_induced_price(row, current_price):
    """
    Calculate induced price by reducing margin by one step.
    Used for Zero Demand and High DOH scenarios.
    
    Returns:
        Induced price if valid and lower than current, else None
    """
    wac_p = float(row.get('wac_p', 0) or 0)
    if wac_p <= 0 or current_price <= 0:
        return None
    
    current_margin = (current_price - wac_p) / current_price
    margin_step = calculate_margin_step(row)
    new_margin = current_margin - margin_step
    
    if new_margin >= 1:
        return None
    
    induced_price = wac_p / (1 - new_margin)
    induced_price = round(induced_price * 4) / 4  # Round to 0.25
    
    # Apply floors: min_induced_price and commercial_min_price
    min_induced = float(row.get('min_induced_price', 0) or 0)
    commercial_min = float(row.get('commercial_min_price', 0) or 0)
        
    floor_price = max(min_induced, commercial_min) if commercial_min > 0 else min_induced
    
    if induced_price < floor_price:
        return None  # Can't reduce further
    
    return induced_price if induced_price < current_price else None

# =============================================================================
# MAIN ENGINE: GENERATE PERIODIC ACTION
# =============================================================================

def generate_periodic_action(row, previous_df):
    """
    Generate periodic action based on UTH performance.
    
    Logic:
    - Zero Demand: 1 step below current + SKU discount
    - On Track: No action
    - Growing: Deactivate discounts or increase price, reduce cart if too open
    - Dropping: Based on qty_ratio vs retailer_ratio:
        - qty OK, retailers dropping: SKU discount (then price if already has)
        - qty dropping, retailers OK: QD (then price if already has)
        - both dropping: SKU discount (then price if already has)
    - Price reduction max 2x per day
    """
    product_id = row.get('product_id')
    warehouse_id = row.get('warehouse_id')
    
    result = {
        'product_id': product_id,
        'warehouse_id': warehouse_id,
        'cohort_id': row.get('cohort_id'),
        'sku': row.get('sku'),
        'brand': row.get('brand'),
        'cat': row.get('cat'),
        'stocks': row.get('stocks', 0),
        'current_price': row.get('current_price'),
        'wac_p': row.get('wac_p'),
        'uth_qty': row.get('uth_qty', 0),
        'uth_retailers': row.get('uth_retailers', 0),
        'p80_daily_240d': row.get('p80_daily_240d', 1),
        'p70_daily_retailers_240d': row.get('p70_daily_retailers_240d', 1),
        'avg_uth_pct': row.get('avg_uth_pct', 0.5),
        # Today's UTH discount contributions
        'sku_disc_cntrb_uth': row.get('sku_disc_cntrb_uth', 0) or 0,
        't1_cntrb_uth': row.get('t1_cntrb_uth', 0) or 0,
        't2_cntrb_uth': row.get('t2_cntrb_uth', 0) or 0,
        't3_cntrb_uth': row.get('t3_cntrb_uth', 0) or 0,
        'uth_status': None,
        'qty_ratio': None,
        'retailer_ratio': None,
        'new_price': None,
        'price_action': None,
        'current_cart_rule':row.get('current_cart_rule'),
        'new_cart_rule': None,
        'activate_sku_discount': False,  # True = SKU should have discount after this run
        'activate_qd': False,             # True = SKU should have QD after this run
        'keep_qd_tiers': None,            # List of QD tiers to keep (e.g., ['T1', 'T2'])
        # QD tier configuration (passed to qd_handler)
        'qd_tier_1_qty': row.get('qd_tier_1_qty', 0) or 0,
        'qd_tier_1_disc_pct': row.get('qd_tier_1_disc_pct', 0) or 0,
        'qd_tier_2_qty': row.get('qd_tier_2_qty', 0) or 0,
        'qd_tier_2_disc_pct': row.get('qd_tier_2_disc_pct', 0) or 0,
        'qd_tier_3_qty': row.get('qd_tier_3_qty', 0) or 0,
        'qd_tier_3_disc_pct': row.get('qd_tier_3_disc_pct', 0) or 0,
        'removed_discount': None,         # Which discount was removed (for Growing)
        'removed_discount_cntrb': 0,      # Contribution of removed discount
        'price_reductions_today': row.get('reduced_count', 0) or 0,
        'action_reason': None,
        # =====================================================================
        # ADDITIONAL COLUMNS FOR QD AND SKU DISCOUNT HANDLERS
        # =====================================================================
        # Pricing and margin data
        'target_margin': row.get('target_margin'),
        'min_boundary': row.get('min_boundary'),
        'doh': row.get('doh', 0),  # Days on hand - for SKU discount handler
        'mtd_qty': row.get('mtd_qty', 0),  # MTD quantity - for QD ranking
        # Active SKU discount info - for SKU discount handler
        'active_sku_disc_pct': row.get('active_sku_disc_pct', 0),
        'has_active_sku_discount': row.get('has_active_sku_discount', 0),
        'has_active_qd': row.get('has_active_qd', 0),
        # Market margins (converted to prices in handlers)
        'below_market': row.get('below_market'),
        'market_min': row.get('market_min'),
        'market_25': row.get('market_25'),
        'market_50': row.get('market_50'),
        'market_75': row.get('market_75'),
        'market_max': row.get('market_max'),
        'above_market': row.get('above_market'),
        # Margin tiers (converted to prices in handlers)
        'margin_tier_below': row.get('margin_tier_below'),
        'margin_tier_1': row.get('margin_tier_1'),
        'margin_tier_2': row.get('margin_tier_2'),
        'margin_tier_3': row.get('margin_tier_3'),
        'margin_tier_4': row.get('margin_tier_4'),
        'margin_tier_5': row.get('margin_tier_5'),
        'margin_tier_above_1': row.get('margin_tier_above_1'),
        'margin_tier_above_2': row.get('margin_tier_above_2'),
    }
    
    # Skip if OOS (price only in Module 2)
    if row.get('stocks', 0) <= 0:
        result['action_reason'] = 'OOS - skip (price only in Module 2)'
        return result
    
    # Skip if below minimum stock (stock < min selling unit qty)
    if row.get('below_min_stock_flag', 0) == 1:
        result['action_reason'] = 'Below min stock - skip (cannot sell)'
        return result
    
    # Count previous price reductions today
    price_reductions_today = row.get('reduced_count', 0)
    can_reduce_price = price_reductions_today < MAX_PRICE_REDUCTIONS_PER_DAY
    # Count previous price increases today (combined: Module 3 + Module 4)
    m3_increase_today = row.get('increase_count', 0)
    m4_increase_today = row.get('m4_increase_count', 0)
    price_increase_today = m3_increase_today + m4_increase_today
    can_increase_price = price_increase_today < MAX_PRICE_REDUCTIONS_PER_DAY    
    
    # Calculate UTH benchmark: in-stock contribution * P80_qty (distribution-weighted in-stock hours)
    # Convert to float to handle decimal.Decimal from Snowflake
    p80_qty = float(row.get('p80_daily_240d', 1) or 1)
    p70_retailers = float(row.get('p70_daily_retailers_240d', 1) or 1)
    uth_perc_fb = float(row.get('avg_uth_pct', 0.5) or 0.5)
    in_stock_contrib_qty = float(row.get('in_stock_contribution_qty', row.get('avg_uth_pct', 0.5)) or row.get('avg_uth_pct', 0.5) or 0.5)
    in_stock_contrib_ret = float(row.get('in_stock_contribution_ret', row.get('avg_uth_pct', 0.5)) or row.get('avg_uth_pct', 0.5) or 0.5)
    qtr_cntrb = float(row.get('qtr_cntrb', 1.0) or 1.0)
    target_qty = row.get('target_qty')
    uth_cntrb = np.minimum(in_stock_contrib_qty, uth_perc_fb)
    p80_target = p80_qty * qtr_cntrb * uth_cntrb
    turnover_target = float(target_qty) * uth_cntrb if pd.notna(target_qty) else 0
    uth_qty_target = max(p80_target, turnover_target, 4)
    uth_retailer_target = np.maximum(p70_retailers * np.minimum(in_stock_contrib_ret,uth_perc_fb),2)
    
    uth_qty = float(row.get('uth_qty', 0) or 0)
    uth_retailers = float(row.get('uth_retailers', 0) or 0)
    
    # Calculate UTH ratios
    qty_ratio = uth_qty / uth_qty_target if uth_qty_target > 0 else 0
    retailer_ratio = uth_retailers / uth_retailer_target if uth_retailer_target > 0 else 0
    
    result['uth_qty_target'] = round(uth_qty_target, 2)
    result['uth_retailer_target'] = round(uth_retailer_target, 2)
    result['qty_ratio'] = round(qty_ratio, 2)
    result['retailer_ratio'] = round(retailer_ratio, 2)
    
    current_price = float(row.get('current_price') or 0)
    current_cart = float(row.get('current_cart_rule', row.get('normal_refill', 10)) or 10)
    has_sku_disc = row.get('has_active_sku_discount', 0) == 1
    has_qd = row.get('has_active_qd', 0) == 1
    
    # Determine if qty/retailers are dropping (below threshold)
    qty_dropping = qty_ratio < UTH_DROPPING_THRESHOLD
    qty_ok = qty_ratio >= UTH_DROPPING_THRESHOLD
    retailer_dropping = retailer_ratio < UTH_DROPPING_THRESHOLD
    retailer_ok = retailer_ratio >= UTH_DROPPING_THRESHOLD
    
    # =========================================================================
    # CASE 1: Zero demand today (uth_qty = 0)
    # - Reduce price ONCE per day + apply SKU discount
    # - If already reduced price today: just keep SKU discount (no additional reduction)
    # - Open cart if tight (both cases)
    # =========================================================================
    closing_stock_yesterday = float(row.get('closing_stock_yesterday', 0) or 0)
    zero_demand_flag = int(row.get('zero_demand', 0) or 0)
    if zero_demand_flag == 1 and closing_stock_yesterday > 0:
        result['uth_status'] = 'Zero Demand'
        result['activate_sku_discount'] = True
        result['activate_qd'] = True  # Add QD for bulk purchase incentive
        
        # Open cart widely for Zero Demand - use layer_3, fallback to 100
        layer_3_value = None
        try:
            cohort_id = row.get('cohort_id')
            product_id = row.get('product_id')
            percentile_row = df_percentiles[
                (df_percentiles['cohort_id'] == cohort_id) & 
                (df_percentiles['product_id'] == product_id)
            ]
            if len(percentile_row) > 0:
                layer_3_value = percentile_row.iloc[0].get('layer_3')
        except:
            pass
        
        if pd.notna(layer_3_value) and float(layer_3_value) > 0:
            new_cart_zero = np.maximum(np.maximum(int(float(layer_3_value)),current_cart),100)
        else:
            new_cart_zero = 150
        
        if new_cart_zero > current_cart:
            result['new_cart_rule'] = new_cart_zero
            cart_action = f' + open cart to {new_cart_zero}'
        else:
            cart_action = ''
        
        # Price reduction: only if SKU already has active discounts (SKU disc or QD) and still 0 demand
        # First time zero demand -> add discounts only. Next time (already has discounts) -> reduce price.
        if can_reduce_price and (has_sku_disc or has_qd):
            induced_price = calculate_induced_price(row, current_price)
            if induced_price:
                result['new_price'] = induced_price
                result['price_action'] = 'zero_demand_price_decrease'
                result['action_reason'] = f'Zero demand - price reduced ({current_price:.2f} -> {induced_price:.2f}) + SKU discount + QD{cart_action}'
            else:
                result['price_action'] = 'add_sku_disc'
                result['action_reason'] = f'Zero demand - no lower price available + SKU discount + QD{cart_action}'
        elif not can_reduce_price:
            result['price_action'] = 'keep_sku_disc'
            result['action_reason'] = f'Zero demand - price already reduced today, keep SKU discount + QD{cart_action}'
        else:
            result['price_action'] = 'add_discounts_first'
            result['action_reason'] = f'Zero demand - activating discounts first (no price reduction yet){cart_action}'
        
        return result
    
    # =========================================================================
    # CASE 1.5: HIGH DOH (responsive_doh > 30) - Two-step approach
    # - If NO existing SKU discount: Add SKU discount ONLY (wait for next day)
    # - If HAS existing SKU discount and qty_ratio >= 0.9 ("grew"): Keep discount only
    # - If HAS existing SKU discount and qty_ratio < 0.9 ("didn't grow"): Keep discount + induced price
    # Only applies if inventory value (stocks * price) > 10,000 EGP
    # Skip SKUs that were out of stock yesterday (oos_yesterday = 1)
    # =========================================================================
    DOH_HIGH_TURNOVER_THRESHOLD = 30
    HIGH_INVENTORY_VALUE_THRESHOLD = 200
    responsive_doh = float(row.get('responsive_doh', 999) or 999)
    stocks = float(row.get('stocks', 0) or 0)
    inventory_value = stocks * current_price
    oos_yesterday = int(row.get('oos_yesterday', 0) or 0)
    
    if responsive_doh > DOH_HIGH_TURNOVER_THRESHOLD and inventory_value > HIGH_INVENTORY_VALUE_THRESHOLD and oos_yesterday != 1:
        result['uth_status'] = 'High DOH'
        result['activate_sku_discount'] = True
        result['activate_qd'] = True  # Add QD for bulk purchase incentive to move inventory faster
        # Open cart widely for High DOH - use layer_3, fallback to 100
        layer_3_value = None
        try:
            cohort_id = row.get('cohort_id')
            product_id = row.get('product_id')
            percentile_row = df_percentiles[
                (df_percentiles['cohort_id'] == cohort_id) & 
                (df_percentiles['product_id'] == product_id)
            ]
            if len(percentile_row) > 0:
                layer_3_value = percentile_row.iloc[0].get('layer_3')
        except:
            pass
        
        if pd.notna(layer_3_value) and float(layer_3_value) > 0:
            new_cart_doh = np.maximum(int(float(layer_3_value)),current_cart)
        else:
            new_cart_doh = 150
        
        if new_cart_doh > current_cart:
            result['new_cart_rule'] = new_cart_doh
            cart_msg = f' + open cart to {new_cart_doh}'
        else:
            cart_msg = ''
        
        if not has_sku_disc:
            # First occurrence: Add SKU discount only - wait for next day
            result['price_action'] = 'add_sku_disc_doh'
            result['action_reason'] = f'High DOH ({responsive_doh:.1f} days) - ADD SKU discount (wait for next day)' + cart_msg
            return result
        
        else:
            # Has existing SKU discount - check if "grew" (qty_ratio >= 0.9)
            if qty_ratio >= 0.9:
                # SKU "grew" - keep discount but don't reduce price
                result['price_action'] = 'keep_sku_disc'
                result['action_reason'] = f'High DOH ({responsive_doh:.1f} days) + grew (qty={qty_ratio:.2f}) - KEEP SKU discount only' + cart_msg
                return result
            else:
                # SKU "didn't grow" - keep discount + reduce price with induced logic
                if can_reduce_price:
                    induced_price = calculate_induced_price(row, current_price)
                    if induced_price:
                        result['new_price'] = induced_price
                        result['price_action'] = 'induced_doh_reduction'
                        result['action_reason'] = f'High DOH ({responsive_doh:.1f} days) + didn\'t grow (qty={qty_ratio:.2f}) - INDUCED price ({current_price:.2f} -> {induced_price:.2f})' + cart_msg
                        return result
                    else:
                        result['price_action'] = 'keep_sku_disc'
                        result['action_reason'] = f'High DOH ({responsive_doh:.1f} days) - no lower price available' + cart_msg
                        return result
                else:
                    result['price_action'] = 'keep_sku_disc'
                    result['action_reason'] = f'High DOH ({responsive_doh:.1f} days) - price reduction limit reached' + cart_msg
                    return result
    
    # =========================================================================
    # CASE 1.6: LOW STOCK PROTECTION (DOH <= 2 with demand)
    # Protect inventory until next receiving - no price reduction, cap cart at normal_refill
    # But still allow price INCREASE if growing
    # =========================================================================
    normal_refill = float(row.get('normal_refill', 5) or 5)
    is_low_stock = responsive_doh <= LOW_STOCK_DOH_THRESHOLD and uth_qty > 0
    
    if is_low_stock:
        result['uth_status'] = 'Low Stock Protected'
        result['price_action'] = 'hold_low_stock'
        
        # Cap cart rule at normal_refill (don't open cart wide for low stock)
        if current_cart > normal_refill:
            result['new_cart_rule'] = np.ceil(max(int(normal_refill),5) + float(row.get('refill_stddev', 2) or 2))
            result['action_reason'] = f'Low stock (DOH={responsive_doh:.1f}) - hold price, cap cart to {int(normal_refill)}'
        else:
            result['action_reason'] = f'Low stock (DOH={responsive_doh:.1f}) - hold price'
        
        # Still allow price INCREASE if growing
        if qty_ratio > UTH_GROWING_THRESHOLD and can_increase_price:
            new_price = find_next_price_above(current_price, row)
            if pd.notna(new_price) and new_price > current_price:
                result['new_price'] = new_price
                result['price_action'] = 'low_stock_increase'
                result['action_reason'] += f' + increase price ({current_price:.2f} -> {new_price:.2f})'
        
        return result
    
    # =========================================================================
    # CASE 2: On Track (both qty and retailers ±10%)
    # If has existing discounts, keep them (they'll be deactivated otherwise)
    # =========================================================================
    if (UTH_DROPPING_THRESHOLD <= qty_ratio <= UTH_GROWING_THRESHOLD and
        UTH_DROPPING_THRESHOLD <= retailer_ratio <= UTH_GROWING_THRESHOLD):
        result['uth_status'] = 'On Track'
        result['price_action'] = 'hold'
        
        # Preserve existing discounts (all discounts are deactivated at start of each run)
        if has_sku_disc:
            result['activate_sku_discount'] = True
            result['action_reason'] = f'On Track (qty={qty_ratio:.2f}, ret={retailer_ratio:.2f}) - keep existing SKU discount'
        elif has_qd:
            result['activate_qd'] = True
            result['action_reason'] = f'On Track (qty={qty_ratio:.2f}, ret={retailer_ratio:.2f}) - keep existing QD'
        else:
            result['action_reason'] = f'On Track (qty={qty_ratio:.2f}, ret={retailer_ratio:.2f}) - no action'
        
        return result
    
    # =========================================================================
    # CASE 2.5: Retailers Growing but Qty On Track
    # Action: Increase price 1 step (high retailer demand, normal qty = opportunity)
    # =========================================================================
    if (UTH_DROPPING_THRESHOLD <= qty_ratio <= UTH_GROWING_THRESHOLD and
        retailer_ratio > UTH_GROWING_THRESHOLD):
        result['uth_status'] = 'Retailers Growing'
        if retailer_ratio > RETAILER_PRICE_INCREASE_THRESHOLD and can_increase_price:
            new_price = find_next_price_above(current_price, row)
            if pd.notna(new_price) and new_price > current_price:
                result['new_price'] = new_price
                result['price_action'] = 'retailers_growing_increase'
                result['action_reason'] = f'Retailers growing (qty={qty_ratio:.2f}, ret={retailer_ratio:.2f}) - increase price ({current_price:.2f} -> {new_price:.2f})'
            else:
                result['price_action'] = 'hold'
                result['action_reason'] = f'Retailers growing (qty={qty_ratio:.2f}, ret={retailer_ratio:.2f}) - no tier above, hold'
        else:
            result['price_action'] = 'hold'
            result['action_reason'] = f'Retailers growing (qty={qty_ratio:.2f}, ret={retailer_ratio:.2f}) - below price increase threshold ({RETAILER_PRICE_INCREASE_THRESHOLD}), hold'
        
        return result
    
    # =========================================================================
    # CASE 3: Growing (qty > 110%)
    # Find discount with HIGHEST contribution (from TODAY's UTH) and remove it
    # Keep (re-activate) the others
    # If no discounts -> increase price
    # =========================================================================
    if qty_ratio > UTH_GROWING_THRESHOLD:
        result['uth_status'] = 'Growing'
        
        # Get TODAY's UTH discount contributions (not yesterday's)
        sku_disc_cntrb = row.get('sku_disc_cntrb_uth', 0) or 0
        t1_cntrb = row.get('t1_cntrb_uth', 0) or 0
        t2_cntrb = row.get('t2_cntrb_uth', 0) or 0
        t3_cntrb = row.get('t3_cntrb_uth', 0) or 0
        
        # Build list of EXISTING discounts with their contributions
        # Note: We check if tiers EXIST (qty > 0), not just if they had sales today
        # A tier can exist but have 0 contribution if no orders used it yet today
        active_discounts = []
        flag_inc = 1
        
        # SKU discount: check if it exists (has_sku_disc from active discount query)
        if has_sku_disc:
            active_discounts.append(('sku_disc', sku_disc_cntrb))  # Include even if cntrb=0
        
        # QD tiers: check if each tier EXISTS (qty > 0 means the tier is configured)
        if has_qd:
            qd_t1_qty = row.get('qd_tier_1_qty', 0) or 0
            qd_t2_qty = row.get('qd_tier_2_qty', 0) or 0
            qd_t3_qty = row.get('qd_tier_3_qty', 0) or 0
            
            if qd_t1_qty > 0:  # Tier 1 exists
                active_discounts.append(('qd_t1', t1_cntrb))  # Include even if cntrb=0
            if qd_t2_qty > 0:  # Tier 2 exists
                active_discounts.append(('qd_t2', t2_cntrb))  # Include even if cntrb=0
            if qd_t3_qty > 0:  # Tier 3 exists
                active_discounts.append(('qd_t3', t3_cntrb))  # Include even if cntrb=0
        
        if active_discounts:
            # Sort by contribution descending - remove the highest
            active_discounts.sort(key=lambda x: x[1], reverse=True)
            highest_disc, highest_cntrb = active_discounts[0]
            if highest_cntrb >= 50:
                flag_inc = 0
            remaining_discounts = [d[0] for d in active_discounts[1:]]
            
            # Determine what to keep (re-activate)
            keep_sku_disc = 'sku_disc' in remaining_discounts
            keep_qd_t1 = 'qd_t1' in remaining_discounts
            keep_qd_t2 = 'qd_t2' in remaining_discounts
            keep_qd_t3 = 'qd_t3' in remaining_discounts
            keep_any_qd = keep_qd_t1 or keep_qd_t2 or keep_qd_t3
            
            # Set activation flags
            if keep_sku_disc:
                result['activate_sku_discount'] = True
            
            if keep_any_qd:
                result['activate_qd'] = True
                result['keep_qd_tiers'] = [t for t in ['T1', 'T2', 'T3'] 
                                           if (t == 'T1' and keep_qd_t1) or 
                                              (t == 'T2' and keep_qd_t2) or 
                                              (t == 'T3' and keep_qd_t3)]
            
            result['removed_discount'] = highest_disc
            result['removed_discount_cntrb'] = highest_cntrb
            result['price_action'] = f'remove_{highest_disc}'
            result['action_reason'] = f'Growing (qty={qty_ratio:.2f}) - remove {highest_disc} (cntrb={highest_cntrb}%)'
            
            if remaining_discounts:
                result['action_reason'] += f', keep {remaining_discounts}'
        
        elif has_sku_disc or has_qd:
            # Has discounts but no contribution data - remove all
            result['price_action'] = 'remove_all_disc'
            result['action_reason'] = f'Growing (qty={qty_ratio:.2f}) - remove all discounts (no contribution data)'
        
        else:
            # No discounts
            result['price_action'] = 'no_discount_growing'
            result['action_reason'] = f'Growing (qty={qty_ratio:.2f}) - no discounts'
        
        # Increase price 1 step only if qty_ratio exceeds stricter price threshold
        
        if can_increase_price and flag_inc and qty_ratio > QTY_PRICE_INCREASE_THRESHOLD:
            new_price = find_next_price_above(current_price, row)
            if pd.notna(new_price) and new_price > current_price:
                result['new_price'] = new_price
                result['action_reason'] += f' + increase price ({current_price:.2f} -> {new_price:.2f})'
            else:
                result['action_reason'] += ' + no tier above for price increase'
        else:
            if not flag_inc:
                result['action_reason'] += ' + Discount removal before increase'
            elif qty_ratio <= QTY_PRICE_INCREASE_THRESHOLD:
                result['action_reason'] += f' + qty_ratio ({qty_ratio:.2f}) below price increase threshold ({QTY_PRICE_INCREASE_THRESHOLD}), hold price'
            else:
                result['action_reason'] += ' + price increase limit reached'
        
        # Reduce cart rule only if qty_per_retailer_ratio = qty_ratio / max(retailer_ratio, 0.01) > 1.3
        # Use percentile-based reduction
        qty_per_retailer_ratio = qty_ratio / max(retailer_ratio, 0.01)
        if qty_per_retailer_ratio > 1.3:
            # Get percentile data for this SKU
            cohort_id = row.get('cohort_id')
            product_id = row.get('product_id')
            percentile_row = df_percentiles[
                (df_percentiles['cohort_id'] == cohort_id) & 
                (df_percentiles['product_id'] == product_id)
            ]
            
            if len(percentile_row) > 0:
                current_level = get_current_percentile_level(current_cart, percentile_row)
                if current_level:
                    next_perc = get_next_lower_percentile(current_level, percentile_row)
                    if pd.notna(next_perc) and next_perc > 0:
                        result['new_cart_rule'] = max(MIN_CART_RULE, min(MAX_CART_RULE, int(round(next_perc))))
                        result['action_reason'] += f' + reduce cart to {int(round(next_perc))} (qty_per_retailer_ratio={qty_per_retailer_ratio:.2f}, percentile-based)'
                    else:
                        result['action_reason'] += f' + cart already at minimum percentile (qty_per_retailer_ratio={qty_per_retailer_ratio:.2f})'
                else:
                    result['action_reason'] += f' + could not determine current percentile level (qty_per_retailer_ratio={qty_per_retailer_ratio:.2f})'
            else:
                result['action_reason'] += f' + no percentile data available for cart reduction (qty_per_retailer_ratio={qty_per_retailer_ratio:.2f})'
        else:
            # Keep current cart rule - qty_per_retailer_ratio at or below tightening threshold
            result['action_reason'] += f' + keep cart (qty_per_retailer_ratio={qty_per_retailer_ratio:.2f} <= 1.3)'
        
        return result
    
    # =========================================================================
    # CASE 4: Dropping - Different actions based on qty vs retailer ratios
    # =========================================================================
    result['uth_status'] = 'Dropping'
    
    def apply_price_reduction():
        """Helper to apply price reduction if allowed."""
        if not can_reduce_price:
            return None, f'Price reduction limit reached ({price_reductions_today}/{MAX_PRICE_REDUCTIONS_PER_DAY} today)'
        
        new_price = find_next_price_below(current_price, row)
        if new_price < current_price:
            commercial_min = float(row.get('commercial_min_price', row.get('minimum', 0)) or 0)
            if pd.notna(commercial_min) and commercial_min > 0:
                new_price = max(new_price, commercial_min)
            return new_price, f'decrease ({current_price:.2f} -> {new_price:.2f})'
        return None, 'no tier below'
    
    # CASE 4A: qty OK (≥90%) but retailers dropping (<90%)
    # Action: SKU discount (add new OR keep existing), then price if already has
    if qty_ok and retailer_dropping:
        # Always set activate_sku_discount = True (either adding new or keeping existing)
        result['activate_sku_discount'] = True
        
        if not has_sku_disc:
            # Adding new SKU discount
            result['price_action'] = 'add_sku_disc'
            result['action_reason'] = f'Retailers dropping (ret={retailer_ratio:.2f}, qty OK) - ADD new SKU discount'
        else:
            # Keeping existing SKU discount + reduce price
            new_price, reason = apply_price_reduction()
            if new_price:
                result['new_price'] = new_price
                result['price_action'] = 'keep_sku_disc_and_decrease'
                result['action_reason'] = f'Retailers dropping - KEEP SKU disc + {reason}'
            else:
                result['price_action'] = 'keep_sku_disc'
                result['action_reason'] = f'Retailers dropping - KEEP SKU disc ({reason})'
    
    # CASE 4B: qty dropping (<90%) but retailers OK (≥90%)
    # Action: QD (add new OR keep existing), then price if already has
    elif qty_dropping and retailer_ok:
        # Always set activate_qd = True (either adding new or keeping existing)
        result['activate_qd'] = True
        
        if not has_qd:
            # Adding new QD
            result['price_action'] = 'add_qd'
            result['action_reason'] = f'Qty dropping (qty={qty_ratio:.2f}, ret OK) - ADD new QD'
        else:
            # Keeping existing QD + reduce price only if ratio meets stricter threshold
            if qty_ratio < QTY_PRICE_DECREASE_THRESHOLD:
                new_price, reason = apply_price_reduction()
                if new_price:
                    result['new_price'] = new_price
                    result['price_action'] = 'keep_qd_and_decrease'
                    result['action_reason'] = f'Qty dropping - KEEP QD + {reason}'
                else:
                    result['price_action'] = 'keep_qd'
                    result['action_reason'] = f'Qty dropping - KEEP QD ({reason})'
            else:
                result['price_action'] = 'keep_qd'
                result['action_reason'] = f'Qty dropping (qty={qty_ratio:.2f}) - KEEP QD, above price decrease threshold ({QTY_PRICE_DECREASE_THRESHOLD})'
    
    # CASE 4C: Both dropping (<90%)
    # Action: SKU discount (add new OR keep existing), then price if already has
    elif qty_dropping and retailer_dropping:
        # Always set activate_sku_discount = True (either adding new or keeping existing)
        result['activate_sku_discount'] = True
        
        if not has_sku_disc:
            # Adding new SKU discount
            result['price_action'] = 'add_sku_disc'
            result['action_reason'] = f'Both dropping (qty={qty_ratio:.2f}, ret={retailer_ratio:.2f}) - ADD new SKU discount'
        else:
            # Keeping existing SKU discount + reduce price only if at least one ratio meets stricter threshold
            if qty_ratio < QTY_PRICE_DECREASE_THRESHOLD or retailer_ratio < RETAILER_PRICE_DECREASE_THRESHOLD:
                new_price, reason = apply_price_reduction()
                if new_price:
                    result['new_price'] = new_price
                    result['price_action'] = 'keep_sku_disc_and_decrease'
                    result['action_reason'] = f'Both dropping - KEEP SKU disc + {reason}'
                else:
                    result['price_action'] = 'keep_sku_disc'
                    result['action_reason'] = f'Both dropping - KEEP SKU disc ({reason})'
            else:
                result['price_action'] = 'keep_sku_disc'
                result['action_reason'] = f'Both dropping (qty={qty_ratio:.2f}, ret={retailer_ratio:.2f}) - KEEP SKU disc, above price decrease thresholds'
    
    else:
        result['price_action'] = 'hold'
        result['action_reason'] = f'Unexpected state (qty={qty_ratio:.2f}, ret={retailer_ratio:.2f})'
    
    # Increase cart for dropping SKUs
    result['new_cart_rule'] = adjust_cart_rule(current_cart, 'increase', row)
    result['action_reason'] += ' + increase cart 20%'
    
    return result

print("Main engine function loaded.")


Main engine function loaded.


In [13]:
df = df.merge(prev_inc,on=['product_id','warehouse_id'],how='left')
df = df.merge(prev_red,on=['product_id','warehouse_id'],how='left')
df['increase_count'] = df['increase_count'].fillna(0)
df['m4_increase_count'] = df['m4_increase_count'].fillna(0)
df['reduced_count'] = df['reduced_count'].fillna(0)

In [14]:
df = df.drop_duplicates(subset=['product_id', 'warehouse_id'], keep='first')
df = df.reset_index() 

In [15]:
# =============================================================================
# EXECUTE MODULE 3
# =============================================================================
print(f"Processing {len(df)} SKUs...")
print("="*60)

results = []
for idx, row in df.iterrows():
    
    result = generate_periodic_action(row, df_previous_actions)
    results.append(result)
    
    if (idx + 1) % 10000 == 0:
        print(f"Processed {idx + 1}/{len(df)} SKUs...")

df_results = pd.DataFrame(results)
print(f"\n✅ Processed {len(df_results)} SKUs")


Processing 29791 SKUs...


Processed 10000/29791 SKUs...


Processed 20000/29791 SKUs...



✅ Processed 29791 SKUs


In [16]:
df_results = df_results.drop_duplicates(subset=['product_id', 'warehouse_id'], keep='first')
print(f"\n✅ Processed {len(df_results)} SKUs")


✅ Processed 29791 SKUs


In [17]:
# =============================================================================
# PRICE FLOOR ENFORCEMENT (excludes Zero Demand and High DOH)
# Floor = lowest price from effective_tiers. Margin can be negative.
# =============================================================================
eligible = (~df_results['uth_status'].isin(['Zero Demand', 'High DOH'])) & (pd.to_numeric(df_results['doh'], errors='coerce').fillna(0) < 30)

def get_floor_price(row):
    tiers = row.get('effective_tiers', [])
    if isinstance(tiers, list) and len(tiers) > 0:
        return tiers[0]
    return np.nan

floor_price = df_results.apply(get_floor_price, axis=1)
floor_price = (floor_price * 4).round() / 4
valid_floor = eligible & floor_price.notna() & (floor_price > 0)

effective_price = df_results['new_price'].combine_first(
    pd.to_numeric(df_results['current_price'], errors='coerce')
)

needs_floor = valid_floor & effective_price.notna() & (effective_price < floor_price)

no_new_price = df_results['new_price'].isna()
current_below = needs_floor & no_new_price
df_results.loc[current_below, 'new_price'] = floor_price[current_below]
df_results.loc[current_below, 'price_action'] = 'price_floor_raise'
df_results.loc[current_below, 'action_reason'] = df_results.loc[current_below].apply(
    lambda r: f"{r['action_reason'] or ''} | Price raised to floor ({r['current_price']:.2f} -> {floor_price[r.name]:.2f})", axis=1
)

new_below = needs_floor & ~no_new_price
df_results.loc[new_below, 'new_price'] = floor_price[new_below]
df_results.loc[new_below, 'action_reason'] = df_results.loc[new_below].apply(
    lambda r: f"{r['action_reason'] or ''} | Price clamped to floor ({floor_price[r.name]:.2f})", axis=1
)

print(f"Price floor enforcement: {needs_floor.sum()} SKUs affected "
      f"({current_below.sum()} raised, {new_below.sum()} clamped)")
print(f"  Excluded: {(~eligible).sum()} Zero Demand / High DOH SKUs")

Price floor enforcement: 0 SKUs affected (0 raised, 0 clamped)
  Excluded: 5130 Zero Demand / High DOH SKUs


In [18]:
# =============================================================================
# FIXED PRICE OVERRIDE (from Google Sheet)
# =============================================================================
df_fixed = get_fixed_prices()
df_results = df_results.merge(df_fixed, on='product_id', how='left')
has_fixed = df_results['fixed_price'].notna()
df_results.loc[has_fixed, 'new_price'] = df_results.loc[has_fixed, 'fixed_price']
df_results.loc[has_fixed, 'price_action'] = 'fixed_price'
df_results.loc[has_fixed, 'action_reason'] = 'Fixed price from Google Sheet'
df_results = df_results.drop(columns=['fixed_price'])
print(f"Fixed price override: {has_fixed.sum()} SKUs set to fixed price from Google Sheet")

# =============================================================================
# FIXED CART RULE OVERRIDE (from Google Sheet - Sheet2)
# =============================================================================
df_fixed_cart = get_fixed_cart_rules()
df_results = df_results.merge(df_fixed_cart, on='product_id', how='left')
has_fixed_cart = df_results['fixed_cart_rule'].notna()
df_results.loc[has_fixed_cart, 'new_cart_rule'] = df_results.loc[has_fixed_cart, 'fixed_cart_rule'].astype(int)
df_results = df_results.drop(columns=['fixed_cart_rule'])
print(f"Fixed cart rule override: {has_fixed_cart.sum()} SKUs set to fixed cart rule from Google Sheet")

Fetching fixed prices from Google Sheet...


/home/ec2-user/service_account_key.json


  Loaded 11 fixed price SKUs
Fixed price override: 113 SKUs set to fixed price from Google Sheet
Fetching fixed cart rules from Google Sheet...


/home/ec2-user/service_account_key.json


  Loaded 0 fixed cart rule SKUs
Fixed cart rule override: 0 SKUs set to fixed cart rule from Google Sheet


In [19]:
# =============================================================================
# SUMMARY
# =============================================================================
print("="*60)
print("MODULE 3 SUMMARY")
print("="*60)

print(f"\nTotal SKUs: {len(df_results)}")

print(f"\nBy UTH Status:")
print(df_results['uth_status'].value_counts(dropna=False).to_string())

# Actions breakdown
price_changes = df_results[df_results['new_price'].notna()]
cart_changes = df_results[df_results['new_cart_rule'].notna()]
sku_disc_activate = df_results[df_results['activate_sku_discount'] == True]
qd_activate = df_results[df_results['activate_qd'] == True]
discounts_removed = df_results[df_results['removed_discount'].notna()]

print(f"\nActions:")
print(f"  Price changes: {len(price_changes)}")
print(f"  Cart rule changes: {len(cart_changes)}")
print(f"  SKU discounts to activate: {len(sku_disc_activate)}")
print(f"  QD to activate: {len(qd_activate)}")
print(f"  Discounts removed (Growing SKUs): {len(discounts_removed)}")


MODULE 3 SUMMARY

Total SKUs: 29791

By UTH Status:
uth_status
None                   12901
Dropping               10638
High DOH                3463
Zero Demand             1277
Growing                  720
Low Stock Protected      567
Retailers Growing        123
On Track                 102

Actions:
  Price changes: 4935
  Cart rule changes: 11970
  SKU discounts to activate: 14818
  QD to activate: 5407
  Discounts removed (Growing SKUs): 194


In [20]:
# =============================================================================
# EXPORT RESULTS
# =============================================================================
output_cols = [
    # Identifiers
    'product_id', 'warehouse_id', 'cohort_id', 'sku', 'brand', 'cat', 'stocks',
    # Pricing data
    'current_price', 'wac_p', 'new_price',
    'target_margin', 'min_boundary',
    # Performance data
    'uth_qty', 'uth_retailers',
    'p80_daily_240d', 'p70_daily_retailers_240d', 'avg_uth_pct',
    'sku_disc_cntrb_uth', 't1_cntrb_uth', 't2_cntrb_uth', 't3_cntrb_uth',
    'uth_qty_target', 'uth_retailer_target', 'qty_ratio', 'retailer_ratio', 'uth_status',
    'doh', 'mtd_qty',
    # Cart rules
    'price_action', 'current_cart_rule', 'new_cart_rule',
    # SKU Discount fields
    'activate_sku_discount', 'active_sku_disc_pct', 'has_active_sku_discount',
    # QD fields (for qd_handler)
    'activate_qd', 'keep_qd_tiers', 'has_active_qd',
    'qd_tier_1_qty', 'qd_tier_1_disc_pct',
    'qd_tier_2_qty', 'qd_tier_2_disc_pct',
    'qd_tier_3_qty', 'qd_tier_3_disc_pct',
    # Market margins (for handlers to convert to prices)
    'below_market', 'market_min', 'market_25', 'market_50',
    'market_75', 'market_max', 'above_market',
    # Margin tiers (for handlers to convert to prices)
    'margin_tier_below', 'margin_tier_1', 'margin_tier_2', 'margin_tier_3', 'margin_tier_4',
    'margin_tier_5', 'margin_tier_above_1', 'margin_tier_above_2',
    # Action tracking
    'removed_discount', 'removed_discount_cntrb',
    'price_reductions_today', 'action_reason'
]

# Filter to existing columns
output_cols = [c for c in output_cols if c in df_results.columns]

# Drop duplicates before saving
df_output = df_results[output_cols].drop_duplicates(subset=['product_id', 'warehouse_id'], keep='first')
# Save df_output state before any manipulation for Slack upload later
temp_df_for_slack = df_output.copy()
print(f"\n✅ Saved {len(temp_df_for_slack)} rows for Slack upload")
print(f"Total records: {len(df_output)} (after removing {len(df_results) - len(df_output)} duplicates)")



✅ Saved 29791 rows for Slack upload
Total records: 29791 (after removing 0 duplicates)


In [21]:
# =============================================================================
# PUSH CART RULES & PRICES
# =============================================================================
# Push cart rules FIRST, then prices
# If cart rules fail for certain cohorts, skip those cohorts for prices

%run push_cart_rules_handler.ipynb
%run push_prices_handler.ipynb
pus = get_packing_units()

# ⚠️ MODE CONFIGURATION:
# - 'testing' (default): Prepare files but DON'T upload to API
# - 'live': Prepare files AND upload to MaxAB API
PUSH_MODE = 'live'  # Change to 'live' when ready to push

# =============================================================================
# STEP 1: Push Cart Rules First
# =============================================================================
print("\n" + "="*70)
print("STEP 1: PUSHING CART RULES")
print("="*70)

cart_result = push_cart_rules(df_output, pus, source_module='module_3', mode=PUSH_MODE)

print(f"\n{'='*60}")
print("CART RULES RESULT")
print(f"{'='*60}")
print(f"Mode: {cart_result['mode']}")
print(f"Cart rule changes: {cart_result['cart_rule_changes']}")
print(f"Pushed: {cart_result['pushed']}")
print(f"Failed: {cart_result['failed']}")
if cart_result['failed_cohorts']:
    print(f"⚠️ Failed cohorts: {cart_result['failed_cohorts']}")

# =============================================================================
# STEP 2: Push Prices (skip failed cohorts)
# =============================================================================
print("\n" + "="*70)
print("STEP 2: PUSHING PRICES")
print("="*70)

# Get failed cohorts from cart rules to skip in price push
failed_cohorts = cart_result.get('failed_cohorts', [])

# Call push_prices with the results, skipping failed cohorts
push_result = push_prices(df_output, pus, source_module='module_3', mode=PUSH_MODE, skip_cohorts=failed_cohorts)

print(f"\n{'='*60}")
print("PRICES RESULT")
print(f"{'='*60}")
print(f"Mode: {push_result['mode']}")
print(f"Source: {push_result['source_module']}")
print(f"Timestamp: {push_result['timestamp']}")
print(f"Total received: {push_result['total_received']}")
print(f"Price changes: {push_result['price_changes']}")
print(f"Pushed: {push_result['pushed']}")
print(f"Skipped: {push_result['skipped']}")
print(f"Failed: {push_result['failed']}")
if push_result.get('skipped_cohorts'):
    print(f"⚠️ Skipped cohorts (cart rules failed): {push_result['skipped_cohorts']}")


Push Cart Rules Handler loaded at 2026-04-18 23:11:04 Cairo time


✓ API credentials loaded successfully


Push Prices Handler loaded at 2026-04-18 23:11:05 Cairo time


✓ API credentials loaded successfully


✓ Google Sheets client initialized
Fetching packing_units ...


...

  Split into 1 chunks (size: 4000)


  Saving chunks:   0%|          | 0/1 [00:00<?, ?it/s]

  Saving chunks: 100%|██████████| 1/1 [00:00<00:00, 40.47it/s]

  Uploading...


    ✓ Chunk 1 uploaded successfully

🚀 UPLOAD COMPLETE
Mode: live
Total prepared: 10443
Total failed: 0
  Cleanup: removed 30 temporary .xlsx files from 2 folder(s)

PRICES RESULT
Mode: live
Source: module_3
Timestamp: 2026-04-18 23:14:11
Total received: 29791
Price changes: 4644
Pushed: 10443
Skipped: 25147
Failed: 0


In [22]:
# =============================================================================
# STEP 3: PROCESS SKU DISCOUNTS
# =============================================================================
# This step handles SKU discounts for SKUs that need them based on UTH performance.
# Market data has already been refreshed, so we pass the df_output directly.

print("\n" + "="*70)
print("STEP 3: PROCESSING SKU DISCOUNTS")
print("="*70)

%run sku_discount_handler.ipynb

# Filter to SKUs that need SKU discount
df_sku_discount = df_results[df_results['activate_sku_discount'] == True].copy()
print(f"SKUs needing SKU discount: {len(df_sku_discount)}")

# Merge market margins and margin tiers from df (not in df_results)
sku_discount_extra_cols = [
    'product_id', 'warehouse_id',
    # Market margins
    'below_market', 'market_min', 'market_25', 'market_50', 
    'market_75', 'market_max', 'above_market',
    # Margin tiers
    'margin_tier_below', 'margin_tier_1', 'margin_tier_2', 'margin_tier_3', 
    'margin_tier_4', 'margin_tier_5', 'margin_tier_above_1', 'margin_tier_above_2',
    # V2 price tiers
    'effective_tiers', 'price_tiers',
    # Other needed columns
    'doh', 'zero_demand', 'target_margin', 'min_boundary', 'active_sku_disc_pct'
]
# Filter to columns that exist in df
sku_discount_extra_cols = [c for c in sku_discount_extra_cols if c in df.columns]

# Merge the extra columns from df
df_sku_discount = df_sku_discount.merge(
    df[sku_discount_extra_cols].drop_duplicates(subset=['product_id', 'warehouse_id']),
    on=['product_id', 'warehouse_id'],
    how='left',
    suffixes=('', '_from_df')
)
print(f"  Merged market margins and margin tiers from df")

if len(df_sku_discount) > 0:
    sku_discount_result = process_sku_discounts(df_sku_discount, mode=PUSH_MODE)
    
    print(f"\n{'='*60}")
    print("SKU DISCOUNT RESULT")
    print(f"{'='*60}")
    print(f"Mode: {sku_discount_result['mode']}")
    print(f"Total input: {sku_discount_result['total_input']}")
    print(f"SKUs to activate: {sku_discount_result['to_activate']}")
    print(f"Deactivated: {sku_discount_result['deactivated']}")
    print(f"Created: {sku_discount_result['created']}")
    print(f"Failed: {sku_discount_result['failed']}")
else:
    print("No SKUs need SKU discounts")

# =============================================================================
# STEP 4: PROCESSING QUANTITY DISCOUNTS (QD)
# =============================================================================
# This step handles QD adjustments for SKUs flagged by the action engine.
# Only processes SKUs where activate_qd=True and uses keep_qd_tiers to determine
# which tiers to maintain.

print("\n" + "="*70)
print("STEP 4: PROCESSING QUANTITY DISCOUNTS")
print("="*70)

%run qd_handler.ipynb

# Filter to SKUs that need QD processing
df_qd = df_results[df_results['activate_qd'] == True].copy()
print(f"SKUs needing QD processing: {len(df_qd)}")

# Required columns for QD handler
# Include all data needed for tier quantity and price calculations
qd_columns = [
    # Identifiers
    'product_id', 'warehouse_id', 'cohort_id', 'sku', 'brand', 'cat',
    # Pricing data
    'wac_p', 'current_price', 'new_price', 'target_margin', 'min_boundary',
    # Cart rules
    'current_cart_rule', 'new_cart_rule',
    # Market margins (to be converted to prices)
    'below_market', 'market_min', 'market_25', 'market_50',
    'market_75', 'market_max', 'above_market',
    # Margin tiers (to be converted to prices)
    'margin_tier_1', 'margin_tier_2', 'margin_tier_3', 'margin_tier_4',
    'margin_tier_5', 'margin_tier_above_1', 'margin_tier_above_2',
    # Performance data (for top SKU selection)
    'mtd_qty',
    # Stock data (for stock value ranking: stocks * wac_p)
    'stocks',
    # V2 price tiers
    'effective_tiers', 'price_tiers',
    # QD configuration
    'keep_qd_tiers'
]
# Filter to columns that exist in df_results
qd_columns = [c for c in qd_columns if c in df_results.columns]
df_qd = df_qd[qd_columns].copy()

# Merge effective_tiers from df (not in df_results)
if 'effective_tiers' in df.columns:
    df_qd = df_qd.merge(
        df[['product_id', 'warehouse_id', 'effective_tiers', 'price_tiers']].drop_duplicates(subset=['product_id', 'warehouse_id']),
        on=['product_id', 'warehouse_id'], how='left'
    )

if len(df_qd) > 0:
    qd_result = process_qd(df_qd, False)
    
    print(f"\n{'='*60}")
    print("QD PROCESSING RESULT")
    print(f"{'='*60}")
    print(f"Mode: {qd_result['mode']}")
    print(f"Total input: {qd_result['total_input']}")
    print(f"Processed: {qd_result['processed']}")
    print(f"Failed: {qd_result['failed']}")
else:
    print("No SKUs need QD processing")

# =============================================================================
# FINAL SUMMARY
# =============================================================================
print("\n" + "="*70)
print("MODULE 3 EXECUTION COMPLETE")
print("="*70)
print(f"Total SKUs processed: {len(df_output)}")
print(f"Price changes: {(df_output['new_price'] != df_output['current_price']).sum()}")
print(f"Cart rule changes: {(df_output['new_cart_rule'] != df_output['current_cart_rule']).sum()}")
print(f"SKUs with SKU discount: {df_output['activate_sku_discount'].sum()}")
print(f"SKUs with QD: {df_output['activate_qd'].sum()}")
print(f"Output saved to: {OUTPUT_FILE}")



STEP 3: PROCESSING SKU DISCOUNTS


/home/ec2-user/.Renviron
/home/ec2-user/service_account_key.json


SKU Discount Handler loaded at 2026-04-18 23:17:52 Cairo time
Excluded categories: []
Excluded brands: []
AWS & API functions defined ✓
✓ API credentials loaded successfully


Snowflake timezone: America/Los_Angeles
Function 1: deactivate_active_sku_discounts() defined ✓


/home/ec2-user/.Renviron
/home/ec2-user/service_account_key.json


...

    ✓ Cohort 704: 260 rules uploaded


    ✓ Cohort 1123: 159 rules uploaded


    ✓ Cohort 1124: 160 rules uploaded


    ✓ Cohort 1125: 161 rules uploaded


    ✓ Cohort 1126: 160 rules uploaded
  Cleanup: removed 10 temporary .xlsx files from 1 folder(s)

  Cart Rules Result:
    Cohorts updated: 9
    Cohorts failed: 0

QD HANDLER - SUMMARY
Mode: LIVE
Total SKUs in input: 5407
SKUs with valid T1 & T2 prices: 5144
SKUs with valid T3 prices: 2114
SKUs after keep_qd_tiers & 400 tier limit: 1860
Total tier entries: 4789
Valid QD configs: 1860
QD found active: 12
QD deactivated: 12
QD created: 1860
QD creation failed: 0
Cart rules updated: 1743 products

QD PROCESSING RESULT
Mode: live
Total input: 5407
Processed: 1860
Failed: 0

MODULE 3 EXECUTION COMPLETE
Total SKUs processed: 29791
Price changes: 29500
Cart rule changes: 29559
SKUs with SKU discount: 14818
SKUs with QD: 5407
Output saved to: module_3_output_20260418_2303.xlsx


In [23]:
x = df_output.copy()

In [24]:
# =============================================================================
# UPLOAD RESULTS TO SNOWFLAKE AND SEND SLACK NOTIFICATION
# =============================================================================
from common_functions import upload_dataframe_to_snowflake, send_text_slack, send_file_slack

# Add created_at as TIMESTAMP (module runs multiple times per day)
df_output = df_output.drop(columns=['keep_qd_tiers','qtr_cntrb'], errors='ignore')
df_output['keep_qd_tiers'] = np.nan
df_output['created_at'] = datetime.now(CAIRO_TZ).replace(second=0, microsecond=0)
# Upload to Snowflake
print("\n" + "="*60)
print("UPLOADING RESULTS TO SNOWFLAKE")
print("="*60)

upload_status = upload_dataframe_to_snowflake(
    "Egypt", 
    df_output, 
    "MATERIALIZED_VIEWS", 
    "pricing_periodic_push", 
    "append", 
    auto_create_table=True, 
    conn=None
)

# Prepare status variables
prices_pushed = push_result.get('pushed', 0) if 'push_result' in dir() else 0
prices_failed = push_result.get('failed', 0) if 'push_result' in dir() else 0
cart_rules_pushed = cart_result.get('pushed', 0) if 'cart_result' in dir() else 0
cart_rules_failed = cart_result.get('failed', 0) if 'cart_result' in dir() else 0

# SKU discount status
sku_disc_processed = len(df_sku_discount) if 'df_sku_discount' in dir() else 0

# QD status
qd_processed = qd_result.get('processed', 0) if 'qd_result' in dir() and qd_result else 0
qd_failed = qd_result.get('failed', 0) if 'qd_result' in dir() and qd_result else 0
df_output.columns = df_output.columns.str.lower()
if upload_status:
    slack_message = f"""✅ *Module 3 - Periodic Actions Completed*

📅 Date: {datetime.now(CAIRO_TZ).strftime('%Y-%m-%d')}
⏰ Completed at: {datetime.now(CAIRO_TZ).strftime('%H:%M:%S')} Cairo time
🔧 Mode: {PUSH_MODE.upper()}

📊 *Results:*
• Total SKUs processed: {len(df_output):,}
• Price changes: {(df_output['new_price'] != df_output['current_price']).sum():,}
• Induced DOH prices: {(df_output['price_action'] == 'induced_doh_reduction').sum():,}
• Cart rule changes: {(df_output['new_cart_rule'] != df_output['current_cart_rule']).sum():,}

📤 *Push Status:*
• 💰 Prices: ✅ {prices_pushed} pushed | ❌ {prices_failed} failed
• 🛒 Cart Rules: ✅ {cart_rules_pushed} pushed | ❌ {cart_rules_failed} failed
• 🏷️ SKU Discounts: {sku_disc_processed} processed
• 📦 Quantity Discounts: ✅ {qd_processed} processed | ❌ {qd_failed} failed

🗄️ Results uploaded to: MATERIALIZED_VIEWS.pricing_periodic_push"""
    
    send_text_slack('new-pricing-logic', slack_message)
    print("✅ Slack notification sent!")
    
    # Send output file to Slack after the text message (using saved copy before manipulation)
    SLACK_CHANNEL_ID = 'C0AAWK97Z3Q'
    send_file_slack(
        temp_df_for_slack, 
        f'📎 Module 3 Output: {len(temp_df_for_slack)} SKUs processed', 
        SLACK_CHANNEL_ID,
        filename=f'module3_periodic_{datetime.now(CAIRO_TZ).strftime("%Y%m%d_%H%M")}.xlsx'
    )
    print("✅ Output file sent to Slack")
    
    print(f"✅ {len(df_output)} records uploaded to Snowflake")
else:
    error_message = f"""❌ *Module 3 - Periodic Actions Failed*

📅 Date: {datetime.now(CAIRO_TZ).strftime('%Y-%m-%d')}
⏰ Failed at: {datetime.now(CAIRO_TZ).strftime('%H:%M:%S')} Cairo time
⚠️ Upload to Snowflake failed - please check logs

📤 *Push Status (before upload failure):*
• 💰 Prices: ✅ {prices_pushed} pushed | ❌ {prices_failed} failed
• 🛒 Cart Rules: ✅ {cart_rules_pushed} pushed | ❌ {cart_rules_failed} failed
• 🏷️ SKU Discounts: {sku_disc_processed} processed
• 📦 Quantity Discounts: ✅ {qd_processed} processed | ❌ {qd_failed} failed"""
    
    send_text_slack('new-pricing-logic', error_message)
    print("❌ Error notification sent to Slack!")
    
    # Still send output file even on error for debugging (using saved copy before manipulation)
    send_file_slack(
        temp_df_for_slack, 
        f'⚠️ Module 3 ERROR: {len(temp_df_for_slack)} SKUs', 
        SLACK_CHANNEL_ID,
        filename=f'module3_periodic_ERROR_{datetime.now(CAIRO_TZ).strftime("%Y%m%d_%H%M")}.xlsx'
    )
    print("✅ Error file sent to Slack")



UPLOADING RESULTS TO SNOWFLAKE


/home/ec2-user/service_account_key.json


/home/ec2-user/service_account_key.json


Message Sent
✅ Slack notification sent!


/home/ec2-user/service_account_key.json


File module3_periodic_20260418_2324.xlsx sent to Slack
✅ Output file sent to Slack
✅ 29791 records uploaded to Snowflake
